In [ ]:
library(SpaMTP) #This is a custom package that has been downloaded from: devtools::install_github("agc888/SpaMTP")

#General Libraries
library(Cardinal)
library(Seurat)

#For plotting + DE plots
library(ggplot2)
#library(ggVennDiagram)
library(EnhancedVolcano)
library(viridis)
library(tidyr)
library(dplyr)
library(openxlsx)
### Specifiy data directory paths where data is stored

DATA_DIR <- "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/raw_data/MALDI/DHB/"
OUT_DIR  <- "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/processed_data/DHB/"
VIS_DATA_DIR <- "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/raw_data/Visium/"
options(repr.plot.width = 20, repr.plot.height = 20)

# Helper functions

In [ ]:
verbose_message <- function(message_text, verbose) {
  if (verbose) {
    message(message_text)
  }
}


FindRegionalPathways <- function(SpaMTP,
                                ident,
                                DE.list,
                                analyte_types = c("genes", "metabolites"),
                                SM_assay = "SPM",
                                ST_assay = "SPT",
                                SM_slot = "counts",
                                ST_slot = "counts",
                                min_path_size = 5,
                                max_path_size = 500,
                                pval_cutoff_mets = 0.05,
                                pval_cutoff_genes = 0.05,
                                all_genes =FALSE,
                                verbose = TRUE) {
  ## Checks for ident in SpaMTP Object
  if (!(ident %in% colnames(SpaMTP@meta.data))) {
    stop(
      "Ident: ",
      ident,
      " not found in SpaMTP object's @meta.data slot ... Make sure the ident column is in your @metadata and is a factor!"
    )
  }
  cluster_vector = as.factor(SpaMTP@meta.data[[ident]])
  assignment = cluster_vector
  cluster = levels(cluster_vector)
  ## Checks for data in SM and/or ST assay
  if ("genes" %in% analyte_types) {
    st_obj =  LayerData(
              object = SpaMTP,
              assay = ST_assay, # Name of your assay (must be a literal string)
              layer = ST_slot    # The function argument accepts the value of the variable
            )
    if (is.null(st_obj)) {
      stop(
        paste0(
          "No data exists in object[[",
          ST_assay,
          "]][",
          ST_slot,
          "] .. If you are using transcriptomic data with 'genes' in 'analyte_types', please ensure this dataslot exists within your SpaMTP object, else remove 'genes' from analyte_tpes"
        )
      )
    } else{
      gene_matrix = Matrix::t(st_obj)
      if (length(cluster_vector) != nrow(gene_matrix)) {
        stop(
          "Please make sure the input ident is a vector the same length as the number of spots/cells in the gene assay!"
        )
      }
    }
  }
  if ("metabolites" %in% analyte_types) {
    sm_obj =  LayerData(
              object = SpaMTP,
              assay = SM_assay, # Name of your assay (must be a literal string)
              layer = SM_slot    # The function argument accepts the value of the variable
            )
    if (is.null(sm_obj)) {
      stop(
        paste0(
          "No data exists in object[[",
          SM_assay,
          "]][",
          SM_slot,
          "] .. If you are using metabolic data with 'metabolites' in 'analyte_types', please ensure this dataslot exists within your SpaMTP object, else remove 'metabolites' from analyte_tpes"
        )
      )
    } else{
      mass_matrix = Matrix::t(sm_obj)
      if (length(cluster_vector) != nrow(mass_matrix)) {
        stop(
          "Please make sure the input ident is a vector the same length as the number of spots/cells in the metabolite assay!"
        )
      }
    }
  }
  # (2) Annotation
  if (is.null(SpaMTP@tools$db_3)) {
    stop(
      "@tools$db_3 is empty! No intermediate annotation data saved in SpaMTP object. Please run AnnotateSM() with save.intermediate = TRUE",
      "or save the database by setting filename = '...' and manually assign the annotation dataframe to @tools$db_3 <- [ ..."
    )
  }
  db_3 <- SpaMTP@tools$db_3
  db_3 = db_3 %>%
    tidyr::separate_rows(Isomers_IDs, IsomerNames, sep = "; ")

  verbose_message(message_text = "Query necessary data and establish pathway database" , verbose = verbose)

  db_3 = db_3 %>% dplyr::mutate(inputid = Isomers_IDs) %>%  dplyr::mutate(chem_source_id = inputid)
  rampid = c()
  verbose_message(message_text = "Query db for addtional matching" , verbose = verbose)
  db_3 = merge(chem_props, db_3, by = "chem_source_id")
  ### Adding DE Results
  db_3 = db_3 %>% mutate(mz_name = paste0("mz-", db_3$observed_mz))
  if (length(DE.list) != length(analyte_types)) {
    stop(
      "Number of DE data.frames provided does not match the number of analyte types specified. Please make sure a DE dataframe is provided for each analyte type"
    )
  }
  verbose_message(message_text = "Constructing DE dataframes.... ", verbose = verbose)
  for (i in 1:length(analyte_types)) {
    verbose_message(
      message_text = paste0(
        "Assuming DE.list[",
        i,
        "] contains ",
        analyte_types[i] ,
        " results .... "
      ),
      verbose = verbose
    )
    DE <- DE.list[[i]]
    if (any(c("avg_log2FC", "logFC") %in% colnames(DE)) &&
        any(c("p_val_adj", "FDR") %in% colnames(DE)) &&
        "cluster" %in% colnames(DE) &&
        "gene" %in% colnames(DE)) {
      if ("logFC" %in% colnames(DE)) {
        colnames(DE)[colnames(DE) == "logFC"] <- "avg_log2FC"
      }
      # Rename FDR to p_val_adj if FDR exists
      if ("FDR" %in% colnames(DE)) {
        colnames(DE)[colnames(DE) == "FDR"] <- "p_val_adj"
      }
      if (analyte_types[i] == "metabolites") {
        DE = DE %>% rename(mz_name = gene)
        db_3 = merge(db_3 , DE, by = "mz_name")
        DE.list[[analyte_types[i]]] <- db_3
      } else {
        DE = DE %>% mutate(commonName = toupper(gene))
        source_gene = merge(DE, source_df[which(grepl(source_df$rampId, pattern = "RAMP_G")), ], by = "commonName")
        DE.list[[analyte_types[i]]] <- source_gene
      }
    } else {
      stop(
        "DE dataframe [",
        i,
        "] provided does not have the correct column names ... column names MUST include 'cluster', 'gene', ('avg_log2FC' or 'logFC') and ('p_val_adj' or 'FDR'). Please adjust column names in all DE data.frames to match ..."
      )
    }
  }
  # Get pathway db
  verbose_message(message_text = "Constructing pathway database ..." , verbose = verbose)
  chempathway = merge(analytehaspathway, pathway, by = "pathwayRampId")

  pathway_db = split(chempathway$rampId, chempathway$pathwayName)
  pathway_db = pathway_db[which(!duplicated(tolower(names(pathway_db))))]
  pathway_db = pathway_db[lapply(pathway_db, length) >= min_path_size  &
                            lapply(pathway_db, length) <= max_path_size]

  gc()
  gsea_all_cluster = data.frame()
  all_ranks = list()
  pb3 = txtProgressBar(
    min = 0,
    max = length(cluster),
    initial = 0,
    style = 3
  )
for (i in cluster) {
  i <- as.character(i)
  ranks <- c()
  if (all_genes == TRUE){
  # --- METABOLITES BLOCK ---
  if ("metabolites" %in% analyte_types) {
    DE_met <- DE.list[["metabolites"]]
    # Filter by cluster and significance
    sub_db3 = DE_met[which(as.character(DE_met$cluster) == i), ] %>% 
      dplyr::filter(p_val_adj <= pval_cutoff_mets %||% 0.05) %>% 
      dplyr::filter(!duplicated(ramp_id))
    
    # 1. Handle p-values of 0 to avoid Inf during log transformation
    # We replace 0 with the smallest floating point number (or a tiny constant like 1e-300)
    p_vals_safe <- sub_db3$p_val
    p_vals_safe[p_vals_safe == 0] <- .Machine$double.xmin 
    
    # 2. Calculate Signed Significance Score: sign(log2FC) * -log10(p_val)
    raw_scores <- sign(sub_db3$avg_log2FC) * (-log10(p_vals_safe))
    
    # 3. Scale the new metric (centering at 0)
    met_ranks = scale(raw_scores, center = 0)
    names(met_ranks) = sub_db3$ramp_id
    ranks <- c(ranks, met_ranks)
  }
  
  # --- GENES BLOCK ---
  if ("genes" %in% names(DE.list)) {
    DE_rna <- DE.list[["genes"]]
    # Filter by cluster and significance
    sub_de_gene = DE_rna[which(as.character(DE_rna$cluster) == i), ] %>% 
      dplyr::filter(p_val_adj <= pval_cutoff_genes %||% 0.05) %>% 
      dplyr::filter(!duplicated(rampId))
    
    # 1. Handle p-values of 0
    p_vals_gene_safe <- sub_de_gene$p_val
    p_vals_gene_safe[p_vals_gene_safe == 0] <- .Machine$double.xmin
    
    # 2. Calculate Signed Significance Score
    raw_scores_gene <- sign(sub_de_gene$avg_log2FC) * (-log10(p_vals_gene_safe))
    
    # 3. Scale the new metric
    ranks_gene_vector = scale(raw_scores_gene, center = 0)
    names(ranks_gene_vector) = sub_de_gene$rampId
    
    ranks <- c(ranks, ranks_gene_vector)
  }
}    else{
        if ("metabolites" %in% analyte_types) {

      ## metabolites

      DE_met <- DE.list[["metabolites"]]

      sub_db3 = DE_met[which(as.character(DE_met$cluster) == i), ] %>% dplyr::filter(p_val_adj <= pval_cutoff_mets %||% 0.05) %>% dplyr::filter(!duplicated(ramp_id))

      met_ranks = scale(sub_db3$avg_log2FC, center = 0)

      names(met_ranks) = sub_db3$ramp_id

      ranks <- c(ranks, met_ranks)

    }

    if ("genes" %in% names(DE.list)) {

      ## genes

      DE_rna <- DE.list[["genes"]]

      sub_de_gene = DE_rna[which(as.character(DE_rna$cluster) == i), ] %>% dplyr::filter(p_val_adj <= pval_cutoff_genes %||% 0.05) %>% dplyr::filter(!duplicated(rampId))

      ranks_gene_vector = scale(sub_de_gene$avg_log2FC, center = 0)

      names(ranks_gene_vector) = sub_de_gene$rampId

      # Genes and metabolites

      ranks <- c(ranks, ranks_gene_vector)

    }
    }
  # --- CLEANUP ---
  ranks = ranks[which(!duplicated(names(ranks)))]
  all_ranks[[i]] = ranks[is.finite(ranks)]


    gsea_result <- c()
    if (length(all_ranks[[i]]) > 0) {
      suppressWarnings({
        gsea_result = fgsea::fgsea(
          pathways =  pathway_db,
          stats = all_ranks[[i]],
          minSize = min_path_size,
          maxSize = max_path_size
        )  %>%  dplyr::mutate(Cluster_id = i)
      })

    } else {
      gsea_result <- data.table::data.table(
        pathway = character(0),
        pval = numeric(0),
        padj = numeric(0),
        log2err = numeric(0),
        ES = numeric(0),
        NES = numeric(0),
        size = integer(0),
        leadingEdge = list(),
        Cluster_id = i
      )
    }
    gsea_result = na.omit(gsea_result) %>% filter(!duplicated(pathway))
    short_source = source_df[which((source_df$rampId %in% names(all_ranks[[i]])) &
                                     !duplicated(source_df$rampId)), ]

    addtional_entry = do.call(rbind, lapply(1:nrow(gsea_result), function(x) {
      temp = unique(unlist(gsea_result$leadingEdge[x]))
      if ("metabolites" %in% analyte_types) {
        temp_ref =   sub_db3[which(sub_db3$ramp_id %in% temp), ] %>% dplyr::mutate(adduct_info = paste0(observed_mz, "[", Adduct, "]")) %>% dplyr::filter(!duplicated(adduct_info))
      }
      if ("genes" %in% analyte_types) {
        temp_rna = short_source[which((short_source$rampId %in% temp) &
                                        (grepl(short_source$rampId, pattern = "RAMP_G"))), ]
      }
      return(
        data.frame(
          adduct_info = if("metabolites" %in% analyte_types){paste0(temp_ref$adduct_info, collapse = ";")}else{""},
          leadingEdge_metabolites = if("metabolites" %in% analyte_types){paste0(sub(";.*", "", temp_ref$IsomerNames), collapse = ";")}else{""},
          leadingEdge_metabolites_id = if("metabolites" %in% analyte_types){paste0(temp_ref$chem_source_id, collapse = ";")}else{""},
          leadingEdge_genes = if("genes" %in% analyte_types){paste0(temp_rna$commonName, collapse = ";")}else{""},
          met_regulation = if("metabolites" %in% analyte_types){paste0(ifelse(ranks[which((names(ranks) %in% temp) &
                                                       (grepl(names(ranks), pattern = "RAMP_C")))] >= 0, "↑", "↓"), collapse = ";")}else{""},
          rna_regulation = if("genes" %in% names(DE.list)){paste0(ifelse(ranks[which((names(ranks) %in% temp) &
                                                       (grepl(names(ranks), pattern = "RAMP_G")))] >= 0, "↑", "↓"), collapse = ";")}else{""}
        )
      )
    }))
    gsea_result = cbind(gsea_result , addtional_entry)
    gsea_all_cluster = rbind(gsea_all_cluster, gsea_result)
    setTxtProgressBar(pb3, as.numeric(which(cluster == i)))
  }
  close(pb3)

  gsea_all_cluster <- na.omit(gsea_all_cluster)%>%
    dplyr::mutate(group_importance = sum(abs(NES)))
  colnames(gsea_all_cluster)[1] = "pathwayName"
  gsea_all_cluster = merge(gsea_all_cluster, pathway, by = "pathwayName")
  return(gsea_all_cluster)
}

# Keep only human genes and remove prefix - Working version for Seurat v5
keep_human_genes_only <- function(seurat_obj, 
                                  assay = "SPT", 
                                  human_prefix = "hg38-",
                                  verbose = TRUE) {
  
  # Check if assay exists
  if (!assay %in% names(seurat_obj)) {
    stop("Assay '", assay, "' not found in Seurat object. Available assays: ", 
         paste(names(seurat_obj), collapse = ", "))
  }
  
  # Get current feature names
  all_features <- rownames(seurat_obj[[assay]])
  
  if (verbose) {
    message("Total features before filtering: ", length(all_features))
  }
  
  # Identify human genes
  human_genes_idx <- grepl(paste0("^", human_prefix), all_features)
  human_features <- all_features[human_genes_idx]
  
  if (length(human_features) == 0) {
    stop("No genes found with prefix '", human_prefix, "'. Please check your prefix.")
  }
  
  message("Keeping ", length(human_features), " human genes out of ", 
          length(all_features), " total genes")
  
  # Remove prefix from human gene names
  new_features <- gsub(paste0("^", human_prefix), "", human_features)
  
  if (verbose) {
    message("\nExample genes after prefix removal:")
    print(head(data.frame(
      original = human_features,
      new = new_features
    ), 10))
  }
  
  # Check for duplicates
  if (any(duplicated(new_features))) {
    warning("Duplicate gene names detected after prefix removal:")
    dup_genes <- new_features[duplicated(new_features)]
    message("Duplicated genes: ", paste(head(dup_genes, 10), collapse = ", "))
  }
  
  # Create a mapping from old to new names
  feature_mapping <- setNames(new_features, human_features)
  
  # Get all layers
  layers <- Layers(seurat_obj[[assay]])
  
  if (verbose) {
    message("\nProcessing layers and renaming features...")
  }
  
  # Process each layer: subset and rename simultaneously
  layer_list <- list()
  for (layer in layers) {
    if (verbose) {
      message("Processing layer: ", layer)
    }
    
    # Get layer data
    layer_data <- LayerData(seurat_obj, assay = assay, layer = layer)
    
    # Subset to human genes only
    layer_data_subset <- layer_data[human_features, , drop = FALSE]
    
    # Rename rows
    rownames(layer_data_subset) <- new_features
    
    # Store in list
    layer_list[[layer]] <- layer_data_subset
    
    if (verbose) {
      message("  Layer ", layer, " processed: ", nrow(layer_data_subset), " features")
    }
  }
  
  # Create a new assay with the processed data
  # Start with the counts layer
  new_assay <- CreateAssay5Object(counts = layer_list[["counts"]])
  
  # Add other layers if they exist
  if ("data" %in% names(layer_list)) {
    new_assay <- SetAssayData(new_assay, layer = "data", new.data = layer_list[["data"]])
  }
  
  if ("scale.data" %in% names(layer_list)) {
    new_assay <- SetAssayData(new_assay, layer = "scale.data", new.data = layer_list[["scale.data"]])
  }
  
  # Update variable features if they exist
  var_features <- VariableFeatures(seurat_obj, assay = assay)
  
  if (length(var_features) > 0) {
    if (verbose) {
      message("\nUpdating variable features...")
    }
    
    # Keep only human variable features
    human_var_features <- var_features[var_features %in% human_features]
    
    if (length(human_var_features) > 0) {
      new_var_features <- gsub(paste0("^", human_prefix), "", human_var_features)
      # Only keep those that exist in new feature set
      new_var_features <- new_var_features[new_var_features %in% new_features]
      
      if (length(new_var_features) > 0) {
        VariableFeatures(new_assay) <- new_var_features
        
        if (verbose) {
          message("Updated ", length(new_var_features), " variable features")
        }
      }
    }
  }
  
  # Replace the old assay with the new one
  seurat_obj[[assay]] <- new_assay
  
  if (verbose) {
    message("\n=== Processing Complete ===")
    message("Final feature count: ", nrow(seurat_obj[[assay]]))
    print(seurat_obj[[assay]])
  }
  
  return(seurat_obj)
}

#' Constructs an interactive network for exploring spatial metabolomics and transcriptomics data.
#'
#' @param SpaMTP A `SpaMTP` Seurat object containing spatial metabolomics (SM) and/or spatial transcriptomics (ST) data. If SM data is included, it must be annotated using the `SpaMTP::AnnotateSM()` function.
#' @param ident A character string specifying the cluster identifier used to group regions, corresponding to a column name in the `SpaMTP@meta.data` slot.
#' @param DE.list A list containing differential expression results from the `FindAllMarkers()` function, with items matching the order of the `analyte_types` argument.
#' @param regpathway A dataframe output from the `SpaMTP::FindRegionalPathways()` function, containing identified regional pathways.
#' @param selected_pathways A character vector specifying the names or IDs of pathways used to construct the network (e.g., `c("Amino acid metabolism", "WP1902", "Aspartate and asparagine metabolism")`). This argument is not case-sensitive.
#' @param path The directory to save the output. If not provided, the default is the current working directory.
#' @param SM_assay A character string specifying the assay name for spatial metabolomics data in `SpaMTP` to extract intensity values (default: `"SPM"`).
#' @param ST_assay A character string specifying the assay name for spatial transcriptomics data in `SpaMTP` to extract RNA count values (default: `"SPT"`).
#' @param SM_slot The slot name containing the spatial metabolomics assay matrix (default: `"counts"`).
#' @param ST_slot The slot name containing the spatial transcriptomics assay matrix (default: `"counts"`).
#' @param colour_palette The color palette used to plot the spatial image in the output HTML file. Default: `grDevices::colorRampPalette(rev(RColorBrewer::brewer.pal(11, "Spectral")))(100)`.
#' @param analyte_types A subset of `c("genes", "metabolites")`. Can be `c("genes")`, `c("metabolites")`, or both.
#' @param image A character vector specifying which images stored within the SpaMTP object to use for plotting (e.g., `c("slice1", "slice2")`). If NULL, all images are used.
#' @param verbose A logical value indicating whether to display detailed messages during execution (default: `FALSE`).
#'
#' @return An interactive HTML file visualizing the network structure of the specified pathways.
#' @export
#'
#' @examples
#' #PathwayNetworkPlots(SpaMTP, ident = "Custom_ident", regpathway = regpathway, DE.list = DE.list, selected_pathways = selected_pathways, image = c("slice1", "slice2"))

PathwayNetworkPlots  = function(SpaMTP,
                                 ident,
                                 regpathway,
                                 DE.list,
                                 selected_pathways = NULL,
                                 path = getwd(),
                                 SM_slot = "counts",
                                 ST_slot = "counts",
                                 colour_palette = NULL,
                                 SM_assay = "SPM",
                                 ST_assay = "SPT",
                                 analyte_types = c("genes", "metabolites"),
                                 image = NULL,
                                 verbose = T) {

  if (is.null(image)) {
    image <- Seurat::Images(SpaMTP)
    if (length(image) == 0) {
      stop("No images found in the Seurat object. Please specify at least one image.")
    }
  }

  if ("genes" %in% analyte_types) {
    if (is.null(SpaMTP@assays[[ST_assay]])) {
      stop(paste0("If you are use genetice data with 'gene' in 'analyte_types' (e.g spatial transcriptolomics), please ensure that '", ST_assay, "' is inside the input seurat object"))
    } else {
      gene_matrix = Matrix::t(SpaMTP[[ST_assay]]@layers[[ST_slot]])
    }
  }
  if ("metabolites" %in% analyte_types) {
    if (is.null(SpaMTP@assays[[SM_assay]])) {
      stop(paste0("If you are use metabolic data with 'metabolites' in 'analyte_types' (e.g spatial metabolomics), please ensure that '", SM_assay, "' is inside the input seurat object"))
    } else {
      mass_matrix = Matrix::t(SpaMTP[[SM_assay]]@layers[[SM_slot]])
    }
  }
  if (("metabolites" %in% analyte_types) & ("genes" %in% analyte_types)) {
    if (ncol(SpaMTP@assays[[SM_assay]]) != ncol(SpaMTP@assays[[ST_assay]])) {
      stop("Please align the spatial data before processing")
    }
  }

  SpatialColors <- grDevices::colorRampPalette(colors = rev(x = RColorBrewer::brewer.pal(n = 11, name = "Spectral")))
  colour_palette = colour_palette %||% SpatialColors(100)

  importance = regpathway %>% dplyr::group_by(pathwayName) %>%
    dplyr::mutate(group_importance = sum(abs(NES))) %>%
    filter(!duplicated(pathwayName)) %>% arrange(desc(group_importance))
  sub_enriched = regpathway[which(
    tolower(regpathway$pathwayName) %in% tolower(selected_pathways %||% importance$pathwayName[1:10]) |
      tolower(regpathway$sourceId) %in% tolower(selected_pathways %||% importance$sourceId[1:10])
  ), ]

  check_topology_existence = function(type, pathway_of_interest, id_of_interest) {
    temp_db=c();if(type=="wiki"){wikiids=unlist(lapply(RAMP_wikipathway,function(x){return(x[["id"]])}));index=which((tolower(names(RAMP_wikipathway))==tolower(pathway_of_interest))|(wikiids==tolower(id_of_interest)));if(length(index)!=0){temp_db=RAMP_wikipathway}}else if(type=="reactome"){reactomeids=unlist(lapply(RAMP_Reactome,function(x){return(x[["id"]])}));index=which((tolower(names(RAMP_Reactome))==tolower(pathway_of_interest))|(reactomeids==tolower(id_of_interest)));if(length(index)!=0){temp_db=RAMP_Reactome}}else if(type=="kegg"){keggids=unlist(lapply(RAMP_kegg,function(x){return(x[["id"]])}));index=which((tolower(names(RAMP_kegg))==tolower(pathway_of_interest))|(keggids==tolower(id_of_interest)));if(length(index)!=0){temp_db=RAMP_kegg}}else if(type=="hmdb"){hmdbids=unlist(lapply(RAMP_hmdb,function(x){return(x[["id"]])}));index=which((tolower(names(RAMP_hmdb))==tolower(pathway_of_interest))|(tolower(hmdbids)==tolower(id_of_interest)));if(length(index)!=0){temp_db=RAMP_hmdb}}else{all_list=c(RAMP_wikipathway,RAMP_Reactome,RAMP_kegg,RAMP_hmdb);all_names=names(all_list);add_ids=unlist(lapply(all_list,function(x){return(x[["id"]])}));index=which((tolower(all_names)==tolower(pathway_of_interest))|(tolower(add_ids)==tolower(id_of_interest)));if(length(index)==0){return(NULL)}else{return(all_list[index[1]])}};if(exists("temp_db")){return(temp_db[[index[1]]])}else{return(NULL)}
  }

  topodb = apply(sub_enriched[!duplicated(sub_enriched$pathwayName), ], MARGIN = 1, FUN = function(x) {
      check_topology_existence(x$type, x$pathwayName, x$sourceId)
  })
  dblengths = unlist(lapply(topodb, length))
  names(topodb) = sub_enriched$pathwayName[!duplicated(sub_enriched$pathwayName)]
  if (any(dblengths == 0)) {
    verbose_message(message_text = paste0("Following pathway(s) cannot find matched topological structure: \n", paste0(sub_enriched[!duplicated(sub_enriched$pathwayName), ]$pathwayName[which(dblengths == 0)], collapse = ";\n")), verbose = verbose)
  }
  topodb = topodb[which(dblengths != 0)]

  for (i in 1:length(analyte_types)) {
    DE.list[[analyte_types[i]]] <- DE.list[[i]]
  }
  
  if (is.null(SpaMTP@tools[["db_3"]])) { verbose_message(message_text = "Please use MZAnnotation " , verbose = verbose) }
  db_3<-SpaMTP@tools$db_3;db_3=db_3%>%tidyr::separate_rows(Isomers,sep=";");input_id=lapply(db_3$Isomers,function(x){x=unlist(x);index_hmdb=which(grepl(x,pattern="HMDB"));x[index_hmdb]=paste0("hmdb:",x[index_hmdb]);index_chebi=which(grepl(x,pattern="CHEBI"));x[index_chebi]=tolower(x[index_chebi]);index_lm=which(grepl(x,pattern="LMPK"));x[index_lm]=tolower(x[index_lm]);return(x)});db_3=db_3%>%dplyr::mutate(inputid=input_id)%>%dplyr::mutate(chem_source_id=input_id);db_3=merge(chem_props,db_3,by="chem_source_id");db_3=db_3%>%mutate(mz_name=paste0("mz-",db_3$observed_mz));

  if (length(DE.list) != length(analyte_types)) {
    stop("Number of DE data.frames provided does not match the number of analyte types specified.")
  }
  
  for (i in 1:length(analyte_types)) {
    DE <- DE.list[[i]]
    if (any(c("avg_log2FC", "logFC") %in% colnames(DE)) && any(c("p_val_adj", "FDR") %in% colnames(DE)) && "cluster" %in% colnames(DE) && "gene" %in% colnames(DE)) {
      if ("logFC" %in% colnames(DE)) { colnames(DE)[colnames(DE) == "logFC"] <- "avg_log2FC" }
      if ("FDR" %in% colnames(DE)) { colnames(DE)[colnames(DE) == "FDR"] <- "p_val_adj" }
      if (analyte_types[i] == "metabolites") {
        DE = DE %>% rename(mz_name = gene)
        db_3 = merge(db_3 , DE, by = "mz_name")
        DE.list[[analyte_types[i]]] <- db_3
      } else {
        DE = DE %>% mutate(commonName = toupper(gene))
        source_gene = merge(DE, source_df[which(grepl(source_df$rampId, pattern = "RAMP_G")), ], by = "commonName")
        DE.list[[analyte_types[i]]] <- source_gene
      }
    } else {
      stop("DE dataframe does not have the correct column names: 'cluster', 'gene', ('avg_log2FC' or 'logFC'), and ('p_val_adj' or 'FDR').")
    }
  }

  sub_enriched = sub_enriched[which(tolower(sub_enriched$pathwayName) %in% tolower(names(topodb))), ]
  pathway_names = unique(sub_enriched$pathwayName)
  network_parts <- c() # <<< FIX: Use a vector to build parts, avoiding comma issues
  ucid = naturalsort::naturalsort(unique(sub_enriched$Cluster_id))
  simplified_content = c()
  matrix_ids = c()
  
  for (i in 1:length(ucid)) {
    sub_cluster = sub_enriched[which(sub_enriched$Cluster_id == ucid[i]), ]
    if ("genes" %in% analyte_types) {
      sub_expr_rna = DE.list[["genes"]][which(tolower(DE.list[["genes"]]$cluster) ==  tolower(ucid[i])), ]
    }
    if ("metabolites" %in% analyte_types) {
      sub_expr_met = DE.list[["metabolites"]][which(tolower(DE.list[["metabolites"]]$cluster) ==  tolower(ucid[i])), ]
    }
    
    cluster_parts <- c() 
    temp_analytes = unique(unlist(lapply(sub_cluster$leadingEdge, function(x) unlist(x))))
    simplified_content = c(simplified_content, temp_analytes)
    
    for (j in 1:length(pathway_names)) {
      if (tolower(pathway_names[j]) %in% tolower(sub_cluster$pathwayName)) {
        temp_topo = topodb[[which(unique(tolower(names(topodb))) == tolower(pathway_names[j]))]]
        path_df = na.omit(
          merge(unique(
            rbind(temp_topo[["mixedEdges"]], temp_topo[["metabolEdges"]], temp_topo[["metabolPropEdges"]], temp_topo[["protPropEdges"]])
          ), as.data.frame(reaction_type), by = "reaction_type") %>% rowwise() %>% mutate(detected = (src %in% temp_analytes) | (dest %in% temp_analytes))
        )
        path_df = path_df %>% add_count(src, name = "src_n") %>% add_count(dest, name = "dest_n")
        pathway_analyte = unique(c(path_df$src, path_df$dest))
        matrix_ids = c(matrix_ids, temp_analytes[which(temp_analytes %in% pathway_analyte)])
        
        temp_nodes = "{nodes: ["
        if (length(pathway_analyte) > 0) {
          for (k in 1:length(pathway_analyte)) {
            current_analyte <- pathway_analyte[k]
            if (is.null(current_analyte) || is.na(current_analyte) || nchar(current_analyte) == 0) next # Skip empty entries
            
            is_detected <- current_analyte %in% temp_analytes
            group_type <- ifelse(grepl(current_analyte, pattern = "RAMP_G"), 'rna', 'mets')
            
            if (is_detected) {
              if (group_type == 'rna') {
                expr_data <- sub_expr_rna[which(sub_expr_rna$rampId == current_analyte), ]
                expr_data <- expr_data[which.min(expr_data$p_val_adj), ]
                expr_val <- ifelse(length(expr_data$avg_log2FC) == 0, '"NA"', expr_data$avg_log2FC)
                name_val <- ifelse(length(expr_data$gene) == 0, 'NA', expr_data$gene)
                display_val <- name_val
              } else {
                expr_data <- sub_expr_met[which(sub_expr_met$ramp_id == current_analyte), ]
                expr_data <- expr_data[which.min(expr_data$p_val_adj), ]
                expr_val <- ifelse(length(expr_data$avg_log2FC) == 0, '"NA"', expr_data$avg_log2FC)
                # <<< FIX 2: Fallback to ID if common_name is missing >>>
                name_val <- ifelse((length(expr_data$common_name) == 0 || is.na(expr_data$common_name)), current_analyte, expr_data$common_name)
                display_val <- paste0(expr_data$mz_name, '[', expr_data$Adduct, ']-', name_val)
              }
              temp_nodes <- paste0(temp_nodes, '{ id:"', current_analyte, '", group:"', group_type, '", expr:', expr_val, ', shape:"', group_type, '", name:"', name_val, '", size:', expr_val, ', display:"', display_val, '"},')
            } else {
              name_val <- source_df$commonName[which(source_df$rampId == current_analyte)][1]
              # <<< FIX 2: Fallback for non-detected analytes >>>
              if (is.na(name_val)) { name_val <- current_analyte }
              temp_nodes <- paste0(temp_nodes, '{ id:"', current_analyte, '", group:"', group_type, '", expr:"Not", shape:"', group_type, '", name:"', name_val, '", size:0.7},')
            }
          }
        }
        temp_nodes <- paste0(temp_nodes, "],")
        
        temp_links <- "links: ["
        if(nrow(path_df) > 0){
          for (z in 1:nrow(path_df)) {
            temp_path_df <- path_df[z, ]
            temp_links <- paste0(temp_links, '{source:"', temp_path_df$src, '", target:"', temp_path_df$dest, '", type:"', temp_path_df$colour, '", weight:"', temp_path_df$dest_n + temp_path_df$src_n, '", style:"', temp_path_df$linetype, '", head:"', temp_path_df$arrowhead, '"},')
          }
        }
        temp_links <- paste0(temp_links, "]}")
        
        cluster_parts <- c(cluster_parts, paste0(temp_nodes, temp_links))
        
      } else {
        cluster_parts <- c(cluster_parts, "{nodes:[], links:[]}")
      }
    }
    network_parts <- c(network_parts, paste0("[", paste(cluster_parts, collapse = ","), "]"))
  }
  network <- paste0("const networks = [", paste(network_parts, collapse=","), "];")
  matrix_ids <- unique(matrix_ids)
  
  tab_div <- paste0('<div class="tab" id="default_tab">', pathway_names[1], "</div>", paste0('<div class="tab">', pathway_names[-1], "</div>", collapse=""))
  options <- paste0('<option value="option1" selected>', ucid[1], "</option>", paste0('<option value="option', 2:length(ucid), '">', ucid[-1], "</option>", collapse=""))
  
  fc_vector <- c()
  if ("genes" %in% analyte_types) { fc_vector <- c(fc_vector, DE.list[["genes"]]$avg_log2FC) }
  if ("metabolites" %in% analyte_types) { fc_vector <- c(fc_vector, DE.list[["metabolites"]]$avg_log2FC) }
  scale_legend <- as.integer(sqrt(max(abs(fc_vector), na.rm = TRUE)))

  # --- MULTI-SAMPLE COORDINATE PROCESSING ---
  all_coords_processed<-list(); all_assignments<-c(); all_spot_names<-c();
  canvas_w<-180; canvas_h<-160; num_samples<-length(image); num_cols<-ceiling(sqrt(num_samples)); num_rows<-ceiling(num_samples/num_cols); cell_w<-canvas_w/num_cols; cell_h<-canvas_h/num_rows;
  full_assignment_df<-SpaMTP@meta.data;
  for(i in seq_along(image)){current_image<-image[i];coords_sample<-Seurat::GetTissueCoordinates(SpaMTP,image=current_image);coords_sample<-na.omit(coords_sample);spot_names_sample<-rownames(coords_sample);assignment_sample<-as.character(full_assignment_df[spot_names_sample,ident]);col_idx<-(i-1)%%num_cols;row_idx<-floor((i-1)/num_cols);x_offset<-col_idx*cell_w;y_offset<-row_idx*cell_h;min_x<-min(coords_sample[,1]);max_x<-max(coords_sample[,1]);min_y<-min(coords_sample[,2]);max_y<-max(coords_sample[,2]);range_x<-max_x-min_x;range_y<-max_y-min_y;if(range_x==0)range_x<-1;if(range_y==0)range_y<-1;scale_factor<-min((cell_w*0.95)/range_x,(cell_h*0.95)/range_y);processed_coords_df<-data.frame(y=((coords_sample[,2]-min_y)*scale_factor)+y_offset,x=((coords_sample[,1]-min_x)*scale_factor)+x_offset);all_coords_processed[[i]]<-processed_coords_df;all_assignments<-c(all_assignments,assignment_sample);all_spot_names<-c(all_spot_names,spot_names_sample)}
  final_coords_df<-do.call(rbind,all_coords_processed);coordi_list<-apply(final_coords_df,1,function(row){paste0('[',row['y'],',',row['x'],']')});coordi<-paste0("const coordinates = [",paste(coordi_list,collapse=","),"];");
  non_na_ind <- all_spot_names;
  # --- END MULTI-SAMPLE PROCESSING ---

  met_plot <- "const metplot = {"
  rna_plot <- "const rnaplot = {"
  for (a in 1:length(matrix_ids)) {
    current_id <- matrix_ids[a]
    if (grepl(current_id, pattern = "RAMP_C")) {
      if ("metabolites" %in% analyte_types) {
        met_temp_sub <- DE.list[["metabolites"]][which(DE.list[["metabolites"]]$ramp_id == current_id), ]
        mz_name <- met_temp_sub$mz_name[which.min(met_temp_sub$p_val_adj)]
        met_mat_index <- which(tolower(colnames(mass_matrix)) == mz_name)
        if (length(met_mat_index) > 0) {
          mmat_return <- mass_matrix[non_na_ind, met_mat_index]
          met_plot <- paste0(met_plot, '"', current_id, '":[', paste0(mmat_return, collapse = ","), '],')
        }
      }
    } else {
      if ("genes" %in% analyte_types) {
        rna_temp_sub <- DE.list[["genes"]]$commonName[which(DE.list[["genes"]]$rampId == current_id)]
        gene_mat_index <- which(tolower(colnames(gene_matrix)) == tolower(unique(rna_temp_sub)[1]))
        if (length(gene_mat_index) > 0) {
          gmat_return <- gene_matrix[non_na_ind, gene_mat_index]
          rna_plot <- paste0(rna_plot, '"', current_id, '":[', paste0(gmat_return, collapse = ","), '],')
        }
      }
    }
  }
  rna_plot <- paste0(rna_plot, "};")
  met_plot <- paste0(met_plot, "};")
  
  cluster_infor <- paste0('const cluster_info = ["', paste0(all_assignments, collapse = '","'), '"]')
  
  html = paste0(
   '
<!DOCTYPE html>
<html lang="en">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Interactive Network Visualization</title>
  <style>
    body {
      display: flex;
      margin: 0;
      font-family: Arial, sans-serif;
    }

    #sidebar {
      width: 200px;
      background-color: #f4f4f4;
      padding: 10px;
      height: 92vh;
      overflow-y: auto;
      border-right: 1px solid #ddd;
    }

    #network-container {
      flex: 1;
      position: relative;
      height: 94vh;
      padding: 0px;
    }

    #network-frame {
      border: 2px solid #333;
      padding: 0px;
      height: 94vh;
      background-color: #f9f9f9;
    }


    .legend {
      top: 10px;
      right: 10px;
      background-color: #fff;
      border: 1px solid #ddd;
      padding: 10px;
      width: 200px;
      height: 92vh;
      position: flex;
    }

    .legend-rectangle {
      width: 90%;
      height: 20px;
      background: linear-gradient(to right, blue, yellow, red);
      margin-bottom: 5px;
      margin: 0 15px;
    }

    .legend-labels {
      display: flex;
      justify-content: space-between;
      /* Space out labels */
      width: 100%;
      /* Match the width of the gradient */
      margin-top: 5px;
      /* Space above the labels */
    }

    .tab {
      padding: 10px;
      cursor: pointer;
      border-bottom: 1px solid #ddd;
    }

    .tab:hover {
      background-color: #eee;
    }

    svg {
      width: 100%;
      height: 100%;
      display: block;
      margin: 5px 0;
    }

    .link {
      stroke-width: 2px;
    }

    .node circle {
      stroke: #fff;
      stroke-width: 1.5px;
    }

    .node text {
      font: 12px sans-serif;
      pointer-events: none;
    }

    .arrowhead {
      fill: none;
      stroke-width: 2px;
    }

    .scale-bar {
      margin: 10px;
    }

    .scale-bar label {
      display: block;
      margin-bottom: 5px;
    }

    .scale-bar input[type="range"] {
      width: 100%;
    }

    .line-legend {
      display: flex;
      align-items: center;
      margin: 5px 0;
    }


    .line-text {
      font-size: 12px;
    }

    .tab {
      padding: 10px;
      cursor: pointer;
    }

    .rastercontainer {
      display: flex;
      align-items: flex-start;
    }

    .rasterCanvas {
      border: 1px solid black;
    }

    .legendCanvas {
      margin-left: 0px;
      padding: 0px;
      border: 1px solid transparent;
    }

    .tab.active {
      background-color: #007BFF;
      color: white;
    }

    .line {
      width: 80px;
      height: 2px;
      background-color: black;
      margin-right: 10px;
    }

    .shape {
      width: 20px;
      height: 20px;
      margin-right: 10px;
      display: inline-block;
    }

    .shape.triangle {
      width: 0;
      height: 0;
      border-left: 10px solid transparent;
      border-right: 10px solid transparent;
      border-bottom: 20px solid black;
    }

    .shape.rectangle {
      background-color: black;
    }

    .shape.circle {
      border-radius: 50%;
      background-color: black;
    }

    p {
      display: inline-block;
      max-width: 200px;
      font-size: 12px;
    }

    .hex1::before {
      width: 0;
      height: 20px;
      border-left: 0px solid transparent;
      border-right: 8px solid transparent;
      border-bottom: 0px solid transparent;
      border-top: -5px solid transparent;
      content: "B22";
      color: black;
      font-size: 35px;
    }

    .slider {
      width: 100px;
      height: 40px;
      background-color: #FFFF00;
      border-radius: 20px;
      position: relative;
      cursor: pointer;
      display: flex;
      justify-content: center;
      align-items: center;
      transition: background-color 0.3s ease;
    }

    .slider.active {
      background-color: #4CAF50;
    }
    .parent {
      display: flex; /* Align children horizontally */
      gap: 10px; /* Add spacing between elements */
      width: 100%;
    }
    .shape-text {
      font-size: 12px;
    }

    .scrollable-select {
      width: 200px;
      height: 50px;
      /* Height determines the visible area */
      overflow-y: auto;
      /* Add vertical scroll */
    }
  </style>
  </head>
  <body>
  <div id="main_all" class = "parent">
  <div id="sidebar">',
    tab_div,
    '
  </div>
  <div id="network-container">
  <div id="network-frame">
  <svg id="network">
  </svg>
  </div>

  </div>

  <div class="legend">
  <h4>Legend: log2 fold change</h4>
  <p><span style="color: red;">●</span> Upregulated</p>
  <p><span style="color: blue;">●</span> Downregulated</p>
  <p><span style="color: gray;">●</span> Not detected</p>

  <div>
  <div class="legend-rectangle" id="colorLegend">
  </div>
  <div class="legend-labels">
  <span>',
    -scale_legend ,
    '</span>
    <span>',
    -scale_legend / 2,
    '</span>
  <span>0</span>
    <span>',
    scale_legend / 2,
    '</span>
    <span>',
    scale_legend ,
    '</span>
  </div>
  </div>
  <hr>


  <div class = "line-legend">
  <canvas id="arrowCanvas1" width="110" height="20"></canvas>
  <div class="line-text">Indirect</div>
  </div>
  <div class = "line-legend">
  <canvas id="arrowCanvas2" width="110" height="20"></canvas>
  <div class="line-text">Binding</div>
  </div>
  <div class = "line-legend">
  <canvas id="arrowCanvas10" width="110" height="20"></canvas>
  <div class="line-text">Dissociation</div>
  </div>

  <div class = "line-legend">
  <canvas id="arrowCanvas3" width="110" height="20"></canvas>
  <div class="line-text">Missing</div>
  </div>
  <div class = "line-legend">
  <canvas id="arrowCanvas4" width="110" height="20"></canvas>
  <div class="line-text">Inhibition</div>
  </div>
  <div class = "line-legend">
  <canvas id="arrowCanvas5" width="110" height="20"></canvas>
  <div class="line-text">Phosphorylationn</div>
  </div>
  <div class = "line-legend">
  <canvas id="arrowCanvas9" width="110" height="20"></canvas>
  <div class="line-text">Dephosphorylation</div>
  </div>


  <div class = "line-legend">
  <canvas id="arrowCanvas6" width="110" height="20"></canvas>
  <div class="line-text">Activation</div>
  </div>
  <div class = "line-legend">
  <canvas id="arrowCanvas7" width="110" height="20"></canvas>
  <div class="line-text">Expression</div>
  </div>
  <div class = "line-legend">
  <canvas id="arrowCanvas12" width="110" height="20"></canvas>
  <div class="line-text">Repression</div>
  </div>

  <div class = "line-legend">
  <canvas id="arrowCanvas8" width="110" height="20"></canvas>
  <div class="line-text">Ubiquitination</div>
  </div>


  <div class = "line-legend">
  <canvas id="arrowCanvas11" width="110" height="20"></canvas>
  <div class="line-text">Methylation</div>
  </div>



  <div class = "line-legend">
  <canvas id="arrowCanvas13" width="110" height="20"></canvas>
  <div class="line-text">Glycosylation</div>
  </div>

  <div class = "line-legend">
  <canvas id="arrowCanvas14" width="110" height="20"></canvas>
  <div class="line-text">State change</div>
  </div>

  <hr>
  <div class="line-legend">
  <div class="shape rectangle"></div>
  <div class="shape-text">Metabolites</div>
  </div>
  <div class="line-legend">
  <div class="shape circle"></div>
  <div class="shape-text">RNA</div>
  </div>
  <hr>
      <button id="saveButton">Save network as svg Image</button>
  </div>
  <div class="scale-bar">
  <label for="node-scale">Node Size:</label>
  <input type="range" id="node-scale" min="0.5" max="3" step="0.1" value="1">
    <label for="link-scale">Link Size:</label>
  <input type="range" id="link-scale" min="0.2" max="2" step="0.05" value="1">
  <div class="slider" id="slider" onclick="toggleState()">Simplified net</div>
  <label for="select1">Select cluster for network:</label>
  <select id="select1" class="scrollable-select" size="5">
',
    options,
    '
    </select>
    <div class="rastercontainer">
      <canvas id="rc1" class="rasterCanvas" width="160" height="180"></canvas>
      <canvas id="lca1" class="legendCanvas" width="40" height="180"></canvas>
    </div>
    <p width="200" display="inline-block" id="spatial_window">Display spatial data by clicking on vertices</p>
    <div class="rastercontainer">
      <canvas id="rc2" class="rasterCanvas" width="160" height="180"></canvas>
    </div>
    <p width="200" display="inline-block" id="clu_window">Cluster 1</p>
  </div>
  </div>
  <script src="https://d3js.org/d3.v7.min.js"></script>
  <script>
  function drawArrowLine(ctx, startX, startY, endX, endY, headLength,angle ) {
    const dx = endX - startX;
    const dy = endY - startY;
    // Draw shaft of the arrow (the main line)
    ctx.beginPath();
    ctx.moveTo(startX, startY);
    ctx.lineTo(endX, endY);
    ctx.stroke();

    // Draw arrowhead]
var arrowAngle2 = -angle;
const arrowX1 = endX - headLength * Math.cos(angle);
const arrowY1 = endY - headLength * Math.sin(angle);
const arrowX2 = endX - headLength * Math.cos(arrowAngle2);
const arrowY2 = endY - headLength * Math.sin(arrowAngle2);

// Draw the first side of the arrowhead
ctx.moveTo(endX, endY);
ctx.lineTo(arrowX1, arrowY1);

// Draw the second side of the arrowhead
ctx.moveTo(endX, endY);
ctx.lineTo(arrowX2, arrowY2);

ctx.stroke();
  }

// Set line width and color

// Draw arrow line with 3 segments: 1 shaft + 2 arrowhead lines
const canvas = document.getElementById("arrowCanvas1");
const ctx1 = canvas.getContext("2d");
ctx1.lineWidth = 2;
ctx1.strokeStyle = "black";
ctx1.setLineDash([8, 10]);
drawArrowLine(ctx1,0, 10, 180-80, 10, 10,Math.PI/4);

const canvas2 = document.getElementById("arrowCanvas2");
const ctx2 = canvas2.getContext("2d");
ctx2.lineWidth = 2;
ctx2.strokeStyle = "blue";
ctx2.setLineDash([]);
drawArrowLine(ctx2,0, 10, 175-80, 10, 10,Math.PI*1.3);


const canvas3 = document.getElementById("arrowCanvas3");
const ctx3 = canvas3.getContext("2d");
ctx3.lineWidth = 2;
ctx3.strokeStyle = "black";
ctx3.setLineDash([]);
drawArrowLine(ctx3,0, 10, 175-80, 10, 8,Math.PI*1.3);
drawArrowLine(ctx3,0, 10, 175-80, 10, 8,Math.PI*0.3);

const canvas4 = document.getElementById("arrowCanvas4");
const ctx4 = canvas4.getContext("2d");
ctx4.lineWidth = 2;
ctx4.strokeStyle = "black";
ctx4.setLineDash([]);
drawArrowLine(ctx4,0, 10, 175-80, 10, 10,Math.PI/2);

const canvas5 = document.getElementById("arrowCanvas5");
const ctx5 = canvas5.getContext("2d");
ctx5.lineWidth = 2;
ctx5.strokeStyle = "orange";
ctx5.setLineDash([]);
drawArrowLine(ctx5,0, 10, 175-80, 10, 10,Math.PI/4);

const canvas6 = document.getElementById("arrowCanvas6");
const ctx6 = canvas6.getContext("2d");
ctx6.lineWidth = 2;
ctx6.strokeStyle = "red";
ctx6.setLineDash([8, 10]);
drawArrowLine(ctx6,0, 10, 175-80, 10, 30,Math.PI/4);

const canvas7 = document.getElementById("arrowCanvas7");
const ctx7 = canvas7.getContext("2d");
ctx7.lineWidth = 2;
ctx7.strokeStyle = "green";
ctx7.setLineDash([8, 10]);
drawArrowLine(ctx7,0, 10, 175-80, 10, 30,Math.PI/4);


const canvas8 = document.getElementById("arrowCanvas8");
const ctx8 = canvas8.getContext("2d");
ctx8.lineWidth = 2;
ctx8.strokeStyle = "purple";
ctx8.setLineDash([8, 10]);
drawArrowLine(ctx8,0, 10, 175-80, 10, 10,Math.PI/4);


const canvas9 = document.getElementById("arrowCanvas9");
const ctx9 = canvas9.getContext("2d");
ctx9.lineWidth = 2;
ctx9.strokeStyle = "orange";
ctx9.setLineDash([8, 10]);
drawArrowLine(ctx9,0, 10, 175-80, 10, 10,Math.PI/4);

const canvas10 = document.getElementById("arrowCanvas10");
const ctx10 = canvas10.getContext("2d");
ctx10.lineWidth = 2;
ctx10.strokeStyle = "black";
ctx10.setLineDash([8, 10]);
drawArrowLine(ctx10,0, 10, 175-80, 10, 10,Math.PI*1.3);

const canvas11 = document.getElementById("arrowCanvas11");
const ctx11 = canvas11.getContext("2d");
ctx11.lineWidth = 2;
ctx11.strokeStyle = "red";
ctx11.setLineDash([]);
drawArrowLine(ctx11,0, 10, 175-80, 10, 10,Math.PI/4);


const canvas12 = document.getElementById("arrowCanvas12");
const ctx12 = canvas12.getContext("2d");
ctx12.lineWidth = 2;
ctx12.strokeStyle = "black";
ctx12.setLineDash([8, 10]);
drawArrowLine(ctx12,0, 10, 175-80, 10, 10,Math.PI/2);

const canvas13 = document.getElementById("arrowCanvas13");
const ctx13 = canvas13.getContext("2d");
ctx13.lineWidth = 2;
ctx13.strokeStyle = "black";
ctx13.setLineDash([]);
drawArrowLine(ctx13,0, 10, 169-80, 10, 10,Math.PI);
ctx13.beginPath();
ctx13.arc(95, 10, 5, 0, Math.PI * 2);
ctx13.stroke();


const canvas14 = document.getElementById("arrowCanvas14");
const ctx14 = canvas14.getContext("2d");
ctx14.lineWidth = 2;
ctx14.strokeStyle = "black";
ctx14.setLineDash([]);
drawArrowLine(ctx14,0, 10,170-70, 10, 10,Math.PI/4);
',
    coordi,
    met_plot,
    '
',
rna_plot,
'
',
cluster_infor,
'
',
paste0(
  'const simplified_elements= ["',
  paste0(simplified_content, collapse = '","'),
  '"]'
),
'
',
paste0("const shrink_ratio_x =", 1),
'
',
paste0("const shrink_ratio_y =", 1),
'
',
paste0(
  'const colour_platte = ["',
  paste0(colour_palette, collapse = '","'),
  '"]'
),
'
// First layer cluster Second layer different pathways
',
network,
'

const colorScale = d3.scaleLinear()
.domain([-',
scale_legend,
', 0, ',
scale_legend,
'])  // Define the input domain
.range(["blue", "yellow", "red"]);
let selectedNetwork1 = 0

  var string_sel = document.getElementById("select1").selectedOptions[0].text
  var rasterCanvas = document.getElementById("rc2");
  var ctx = rasterCanvas.getContext("2d");
  ctx.clearRect(0, 0, rasterCanvas.width, rasterCanvas.height);
    coordinates.forEach((coord, index) => {
      var temp_clu = cluster_info[index];
      if(temp_clu == string_sel.slice(7)|(string_sel == "Baseline" & temp_clu==string_sel)|temp_clu == string_sel){
        var color = "red";
      }else{
        var color = "gray";
      }
      ctx.fillStyle = color;
      ctx.fillRect(coord[0], coord[1], shrink_ratio_x, shrink_ratio_y); // Draw a small square (10x10)
    });
    document.getElementById("clu_window").textContent = string_sel;

document.getElementById("select1").addEventListener("change", function (event) {
  if(slider.textContent==="Simplified net"){
    toggleState()
  };
  selectedNetwork1 = event.target.selectedIndex;
  switchNetwork(network_ind, selectedNetwork1, upper = 1, slider.textContent==="Simplified net"?false:true);
  console.log(selectedNetwork1);
  var rasterCanvas = document.getElementById("rc2");
  var ctx = rasterCanvas.getContext("2d");
  string_sel = document.getElementById("select1").selectedOptions[0].text
  ctx.clearRect(0, 0, rasterCanvas.width, rasterCanvas.height);
    coordinates.forEach((coord, index) => {
      var temp_clu = cluster_info[index];
      if(temp_clu == string_sel.slice(7)|(string_sel == "Baseline" & temp_clu==string_sel)|temp_clu == string_sel){
         var color = "red";
      }else{
        var color = "gray";
      }
      ctx.fillStyle = color;
      ctx.fillRect(coord[0], coord[1], shrink_ratio_x, shrink_ratio_y); // Draw a small square (10x10)
    });
    document.getElementById("clu_window").textContent = string_sel;
});
//var svg = d3.select("#network");
const networkContainer = document.getElementById("network-frame");
function checkBounds(d) {
  if (d.x < 0) d.x = 10;
  if (d.x > width) d.x = width - 30;
  if (d.y < 0) d.y = 30;
  if (d.y > height) d.y = height - 10;
}
// Initial dimensions
let width = networkContainer.clientWidth;
let height = networkContainer.clientHeight;


window.addEventListener("resize", () => {
  // Update dimensions on resize
  width = networkContainer.clientWidth;
  height = networkContainer.clientHeight;
  switchNetwork(network_ind, selectedNetwork1, 1)
  //console.log(`New size: ${width}px by ${height}px`);
});

var checked = 0

function updateNetwork(index, partition, upper,full) {
  const svg = d3.select("#network");
  svg.selectAll("*").remove(); // Clear the current network

  svg.attr("width", width)
  .attr("height", height);
  let network_full = networks[partition][index];
  let test = networks[0][0].links.filter(link => simplified_elements.includes(link.source.id) | simplified_elements.includes(link.target.id))
  console.log(test)
  if(full){
    var network = network_full;
  }else{
    if(test.length == 0){
      let filteredLinks = network_full.links.filter(link =>
     simplified_elements.includes(link.source) | simplified_elements.includes(link.target));
     console.log(network_full.links)
     console.log(simplified_elements)
     console.log(filteredLinks)
     let tobeadded = filteredLinks.flatMap(link => [link.source, link.target])
console.log(tobeadded)
     var filteredNodes = network_full.nodes.filter(b => tobeadded.includes(b.id));
    var network = {
  nodes: filteredNodes,
  links: filteredLinks};
    }else{
      let filteredLinks = network_full.links.filter(link =>
     simplified_elements.includes(link.source.id) | simplified_elements.includes(link.target.id));
     console.log(network_full.links)
     console.log(simplified_elements)
     console.log(filteredLinks)
     let tobeadded = filteredLinks.flatMap(link => [link.source.id, link.target.id])
console.log(tobeadded)
     var filteredNodes = network_full.nodes.filter(b => tobeadded.includes(b.id));
    var network = {
  nodes: filteredNodes,
  links: filteredLinks};
    }
  };
console.log(network)
  var container = svg.append("g")
  .attr("width", width)
  .attr("height", height);

  container.append("defs").selectAll("marker")
  .data(["end"])
  .enter().append("marker")
  .attr("id", "end")
  .attr("viewBox", "0 -5 10 10")
  .attr("refX", 10)
  .attr("refY", 0)
  .attr("markerWidth", 6)
  .attr("markerHeight", 6)
  .attr("orient", "auto")
  .append("path")
  .attr("d", "M0,-5L10,0L0,5")
  .attr("fill", "#000");



  var scale = 1;

  // Define arrowheads

  network.links.forEach((link, i) => {
 if(link.head == "arrow"){
    container.append("defs").append("marker")
    .attr("viewBox", "0 -5 10 10")
    .attr("id", `end-arrow${i}`)
    .attr("refX", 13)
    .attr("refY", 0)
    .attr("markerWidth", 5)
    .attr("markerHeight", 5)
    .attr("orient", "auto")
    .append("path")
    .attr("d", "M0,-5L10,0L0,5")
    .attr("fill", link.type);  // Set the arrowhead color to match the stroke
    }
    if(link.head == "crosshead"){
    container.append("defs").append("marker")
    .attr("id", `end-arrow${i}`)
    .attr("refX", 13)
    .attr("refY", 3.5)
    .attr("markerWidth", 7)
    .attr("markerHeight", 7)
    .attr("orient", "auto")
    var marker_sel = d3.select(`#end-arrow${i}`);
    marker_sel.append("line")
    .attr("class", "marker-line")
    .attr("x1","0")
    .attr("y1","0")
    .attr("x2","7")
    .attr("y2","7")
    .attr("stroke", link.type)
    .attr("stroke-width", "1")


    marker_sel.append("line")
    .attr("class", "marker-line")
    .attr("x1","7")
    .attr("y1","0")
    .attr("x2","0")
    .attr("y2","7")
    .attr("stroke", link.type)
    .attr("stroke-width", "1");  // Set the arrowhead color to match the stroke
    }

    if(link.head == "revarrow"){
    container.append("defs").append("marker")
    .attr("id", `end-arrow${i}`)
    .attr("refX", 14)
    .attr("refY", 3.5)
    .attr("markerWidth", 7)
    .attr("markerHeight", 7)
    .attr("orient", "auto")
    var marker_sel = d3.select(`#end-arrow${i}`);
    marker_sel.append("line")
    .attr("class", "marker-line")
    .attr("x1","3.5")
    .attr("y1","3.5")
    .attr("x2","7")
    .attr("y2","7")
    .attr("stroke", link.type)
    .attr("stroke-width", "1")
    //.attr("viewBox", "0 -5 10 10")
    marker_sel.append("line")
    .attr("class", "marker-line")
    .attr("x1","3.5")
    .attr("y1","3.5")
    .attr("x2","7")
    .attr("y2","0")
    .attr("stroke", link.type)
    .attr("stroke-width", "1");  // Set the arrowhead color to match the stroke
    }
    if(link.head == "thead"){
    container.append("defs").append("marker")
    .attr("id", `end-arrow${i}`)
    //.attr("viewBox", "0 -5 10 10")
    .attr("refX", 10)
    .attr("refY", 5)
    .attr("markerWidth", 3)
    .attr("markerHeight", 10)
    .attr("orient", "auto")
    .append("rect")
    .attr("class", "marker-rect")
    .attr("width","10")
    .attr("height","10")
    .attr("fill", link.type);
    }

    if(link.head == "round"){
    container.append("defs").append("marker")
    .attr("id", `end-arrow${i}`)
    //.attr("viewBox", "0 -5 10 10")
    .attr("refX", 9)
    .attr("refY", 5)
    .attr("markerWidth", 10)
    .attr("markerHeight", 10)
    .attr("orient", "auto")
    .append("circle")
    .attr("r", "2")
    .attr("cx","5")
    .attr("cy","5")
    .attr("stroke", link.type)
    .attr("fill", "transparent")
    .attr("stroke-width", "1");
    }
  });


  const simulation = d3.forceSimulation(network.nodes)
  .force("link", d3.forceLink(network.links).id(d => d.id).distance(d =>Math.log10(d.weight)*60))
  .force("charge", d3.forceManyBody().strength(-30))
  .force("center", d3.forceCenter(width / 2, height / 2))
  .force("collide", d3.forceCollide(30).strength(0.3))
  .on("tick", ticked);
  const drag = d3
  .drag()
  .on("start", dragstart)
  .on("drag", draggeddrag);

  function dragstart() {
    d3.select(this).classed("fixed", true);
  }

  function draggeddrag(event, d) {
    d.fx = clamp(event.x, 0, width);
    d.fy = clamp(event.y, 0, height);
    simulation.alpha(1).restart();
  }

  var link = container.append("g")
  .attr("class", "links")
  .selectAll("line")
  .data(network.links)
  .enter().append("line")
  .attr("stroke", d => d.type)
  .attr("stroke-dasharray", d => d.style === "dashed" ? "4 2" : null)
  .attr("marker-end", (d, i) => `url(#end-arrow${i})`) // Use the dynamic marker
  //.attr("marker-end", "url(#end)")
  .attr("stroke-width", 1);

  var node = container
  .selectAll("g.node")
  .data(network.nodes)
  .enter().append("svg:g")
  .attr("class", function (d) {
    if (d.shape === "mets") {
      return "rect node";
    } else if (d.shape === "protein") {
      return "hex node";
    } else if (d.shape === "rna") {
      return "circle node";
    }
  })
  .attr("transform", "translate(0, 0)")
  .attr("fill", d => typeof d.expr === "number" ? colorScale(d.expr) : "gray")
  .call(d3.drag()
        .on("start", dragstarted)
        .on("drag", dragged)
        .on("end", dragended));

  function click(event, d) {
    delete d.fx;
    delete d.fy;
    d3.select(this).classed("fixed", false);
    simulation.alpha(1).restart();
  }
  function clamp(x, lo, hi) {
    return x < lo ? lo : x > hi ? hi : x;
  }

  node.call(drag).on("click", click);

  d3.selectAll(".rect").append("rect")
  .data(network.nodes)
  .attr("width", d => 10 )
  .attr("height", d => 10 );

  d3.selectAll(".circle").append("circle")
  .data(network.nodes)
  .attr("r", d => 6);


  const hexagonPoints = (radius) => {
    const halfWidth = radius * Math.sqrt(3) / 2;
    return `
    0,${-radius}
    ${halfWidth},${-radius / 2}
    ${halfWidth},${radius / 2}
    0,${radius}
    ${-halfWidth},${radius / 2}
    ${-halfWidth},${-radius / 2}`;
  };
  d3.selectAll(".hex").append("polygon")
  .attr("points", function (d) {
    return hexagonPoints(6 )
  });


  node.append("title")
  .text(d => d.name);


  let label = container.append("g")
  .attr("class", "labels")
  .selectAll("text")
  .data(network.nodes)
  .enter().append("text")
  .attr("class", "label")
  .text(d => d.name).attr("font-size", "12px")
  .attr("fill", d => typeof d.expr === "number" ? colorScale(d.expr) : "gray");

  function dragstarted(event, d) {
    delete d.fx;
    delete d.fy;
    d3.select(this).classed("fixed", false);
    simulation.alpha(0.2).restart();
  }

  function dragged(event, d) {
    d3.select(this).attr("cx", d.fx).attr("cy", d.fy);
    d.fx = event.x;
    d.fy = event.y;
  }

  function dragended(event, d) {
    d.fx = null;
    d.fy = null;
  }

  function ticked() {

    link.attr("x1", function (d) {
      checkBounds(d.source);
      return d.source.x;
    })
    .attr("y1", function (d) {
      checkBounds(d.source);
      return d.source.y;
    })
    .attr("x2", function (d) {
      checkBounds(d.target);
      return d.target.x;
    })
    .attr("y2", function (d) {
      checkBounds(d.source);
      return d.target.y;
    }).attr("d", function (d) {

      var x1 = d.source.x,
      y1 = d.source.y,
      x2 = d.target.x,
      y2 = d.target.y,
      dx = x2 - x1,
      dy = y2 - y1,
      dr = Math.sqrt(dx * dx + dy * dy),
      drx = dr,
      dry = dr,
      xRotation = 0,
      largeArc = 0,
      sweep = 1;
      if (x1 === x2 && y1 === y2) {
        xRotation = -45;
        largeArc = 1;
        drx = 30;
        dry = 20;
        x2 = x2 + 1;
        y2 = y2 + 1;
      }
      return "M" + x1 + "," + y1 + "A" + drx + "," + dry + " " + xRotation + "," + largeArc + "," + sweep + " " + x2 + "," + y2;
    });

    node
    .attr("transform", function (d) {
      checkBounds(d);
      return "translate(" + d.x + ", " + d.y + ")";
    });


    label.attr("x", function (d) {
      checkBounds(d);
      return d.x;
    })
    .attr("y", function (d) {
      checkBounds(d);
      return d.y - 10;
    });
  }

  d3.select("#node-scale").on("input", function () {
    scale = +this.value;
    d3.select("#network").selectAll("*")
    .attr("r", 6 * scale)
    .attr("width", 10 * scale)
    .attr("height", 10 * scale)
    .attr("points", function (d) {
      return hexagonPoints(6*scale)
    });
    d3.select("#network2").selectAll("*")
    .attr("r", 6 * scale)
    .attr("width", 10 * scale)
    .attr("height", 10 * scale)
    .attr("points", function (d) {
      return hexagonPoints(6*scale)
    });
    d3.select("#network").selectAll("text")
    .attr("font-size", 12*scale + "px");
    d3.select("#network2").selectAll("text")
    .attr("font-size", 12*scale + "px");

  });


  d3.select("#link-scale").on("input", function () {
    scale = +this.value;
    d3.select("#network").selectAll("*")
    .attr("stroke-width", 1 * scale);
    d3.select("#network2").selectAll("*")
    .attr("stroke-width", 1 * scale);
  });
  node.on("click", function(event, d) {
    console.log(d.id)
    var rasterCanvas = document.getElementById("rc1");
    var ctx = rasterCanvas.getContext("2d");
    var legendCanvas = document.getElementById("lca1");
    var legendCtx = legendCanvas.getContext("2d");
    var legendHeight = legendCanvas.height-10;
    var numSteps = 100; // Number of steps in the gradient
    var numTicks = 6;
    var tickSpacing = legendHeight / (numTicks - 1);
    legendCtx.fillStyle = "black";
    legendCtx.font = "6px Arial";
    legendCtx.textAlign = "right";
    legendCtx.textBaseline = "right";
    ctx.clearRect(0, 0, rasterCanvas.width, rasterCanvas.height);
    legendCtx.clearRect(0, 0, legendCanvas.width, legendCanvas.height);
    console.log("Node clicked:", d);
    if(d.group === "rna"){
      var rna_plot_sub =  rnaplot[d.id]
      document.getElementById("spatial_window").textContent = d.display;
      // Normalize values to range 0-1
      var minValue = 0;
      var maxValue = getPercentileFromObject(rna_plot_sub ,1);

      coordinates.forEach((coord, index) => {
        const normalizedValue = normalize(rna_plot_sub[index],minValue,maxValue);
        const color =colour_platte[Math.floor(normalizedValue*(colour_platte.length))];
        ctx.fillStyle = color;
        ctx.fillRect(coord[0], coord[1], shrink_ratio_x, shrink_ratio_y); // Draw a small square (10x10)
      });
      // Draw color gradient on the legend
      for (let i = 0; i <= numSteps; i++) {
        const normalizedValue = i / numSteps;
        const color = colour_platte[Math.floor(i*(colour_platte.length)/numSteps)]
        legendCtx.fillStyle = color;
        legendCtx.fillRect(28, legendHeight - (i * (legendHeight / numSteps)),28 , (legendHeight / numSteps));
      }

      for (let i = 0; i < numTicks; i++) {
        const yPosition = legendHeight +5 - (i * tickSpacing);
        const valueAtTick = minValue + (i / (numTicks - 1)) * (maxValue - minValue);

        // Draw tick mark
        legendCtx.beginPath();
        legendCtx.moveTo(10, yPosition);
        legendCtx.lineTo(10, yPosition);
        legendCtx.stroke();

        // Draw label
        legendCtx.fillText(valueAtTick.toFixed(1),27.5, yPosition);
      }
    }else{
      var met_plot_sub =  metplot[d.id]
      document.getElementById("spatial_window").textContent = d.display;
      // Normalize values to range 0-1
      var minValue = 0;
      var maxValue = getPercentileFromObject(met_plot_sub,1);

      coordinates.forEach((coord, index) => {
        const normalizedValue = normalize(met_plot_sub[index],minValue,maxValue);
        const color = colour_platte[Math.floor(normalizedValue*(colour_platte.length))];
        ctx.fillStyle = color;
        ctx.fillRect(coord[0], coord[1], shrink_ratio_x, shrink_ratio_y); // Draw a small square (10x10)
      });

      // Draw color gradient on the legend
      for (let i = 0; i <= numSteps; i++) {
        const normalizedValue = i / numSteps;
        const color = colour_platte[Math.floor(i*(colour_platte.length)/numSteps)];
        legendCtx.fillStyle = color;
        legendCtx.fillRect(28, legendHeight - (i * (legendHeight / numSteps)),28 , (legendHeight / numSteps));
      }

      for (let i = 0; i < numTicks; i++) {
        const yPosition = legendHeight +5 - (i * tickSpacing);
        const valueAtTick = minValue + (i / (numTicks - 1)) * (maxValue - minValue);

        // Draw tick mark
        legendCtx.beginPath();
        legendCtx.moveTo(10, yPosition);
        legendCtx.lineTo(10, yPosition);
        legendCtx.stroke();

        // Draw label
        legendCtx.fillText(valueAtTick.toFixed(1),27.5, yPosition);
      }
    }
  });
  //d3.select("#edge-scale").on("input", function() {
    //    var edgeScale = +this.value;
    //    link
    //        .attr("stroke-width", 2 * edgeScale);
    //});
}
var network_ind = 0

function switchNetwork(index, partition, upper,full) {
  document.getElementById("node-scale").value = 1;
  updateNetwork(index, partition, upper,full);
  network_ind = index
}
const slider = document.getElementById("slider");
default_tab.classList.add("active");
document.querySelectorAll(".tab").forEach(tab => {
  tab.addEventListener("click", function () {
    if(slider.textContent==="Simplified net"){
    toggleState()
    };
    document.querySelectorAll(".tab").forEach(t => t.classList.remove("active"));
    this.classList.add("active");
    const index = Array.from(document.querySelectorAll(".tab")).indexOf(this);
    switchNetwork(index, selectedNetwork1, 0,slider.textContent==="Simplified net"?false:true);
    console.log(index)
  });
});
// Initialize the first network
switchNetwork(0, selectedNetwork1, 0, false);

function normalize(value, minValue, maxValue){
  return (value - minValue) / (maxValue - minValue);
}

// Function to map normalized values to gradient colors (from blue to red)
function getGradientColor(value) {
  const r = Math.floor(value * 255); // Red increases with value
  const g = 0;                       // No green for simplicity
  const b = Math.floor((1 - value) * 255); // Blue decreases with value
  return `rgb(${r},${g},${b})`;
}


function flattenArray(arr) {
  let result = [];

  arr.forEach(item => {
    if (Array.isArray(item)) {
      result = result.concat(flattenArray(item)); // Recursively flatten
    } else {
      result.push(item); // Add non-array values
    }
  });

  return result;
}

function getPercentileFromObject(obj,x) {
  let allValues = [];

  // Iterate through each key in the object
  Object.keys(obj).forEach(key => {
    allValues = allValues.concat(obj[key]);
  });

  // Filter out non-numeric or invalid values (optional)
  const validValues = allValues.filter(value => typeof value === "number" && !isNaN(value));

  validValues.sort((a, b) => a - b);
  const index = Math.ceil(x * validValues.length) - 1; // Subtract 1 for 0-based index
  return validValues[index];
}
function click_drag(event, d) {
  delete d.fx;
  delete d.fy;
  d3.select(this).classed("fixed", false);
  simulation.alpha(1).restart();
}
function toggleState() {
  const slider = document.getElementById("slider");

  slider.classList.toggle("active");

  if (slider.classList.contains("active")) {
    slider.innerHTML = "Full net";
    switchNetwork(network_ind, selectedNetwork1,0,true);
    checked = 1;
  } else {
    slider.innerHTML = "Simplified net";
    switchNetwork(network_ind, selectedNetwork1,0,false);
    checked = 1;
  }
}
document.getElementById("saveButton").addEventListener("click", function () {
  const svgElement = document.getElementById("network");
  const serializer = new XMLSerializer();
  const svgString = serializer.serializeToString(svgElement);

  // Create a Blob from the SVG string
  const svgBlob = new Blob([svgString], { type: "image/svg+xml;charset=utf-8" });
  const url = URL.createObjectURL(svgBlob);

  // Create a link element to trigger the download
  const link = document.createElement("a");
  link.href = url;
  link.download = "networks.svg"; // Set the filename
  document.body.appendChild(link);
  link.click(); // Trigger the download
  document.body.removeChild(link); // Clean up
  URL.revokeObjectURL(url); // Free up memory
});
</script>
</body>
</html>')

  returnname = paste0(ident, "_",format(Sys.time(), "%Y_%m_%d_%H_%M_%S_%Z"))
  full_path <- paste0(path, "/", returnname, ".html")
  if (file.access(path, mode = 2) != 0)
  {
    stop("Error: Cannot write to the specified path. Access denied.")
  }
  if (file.exists(full_path))
  {
    warning(paste(
      "Warning: File",
      full_path,
      "already exists and will be overwritten."
    ))
  }
  tryCatch({
    writeLines(html, full_path)
    message(paste("File successfully written to", full_path))
  }, error = function(e) {
    stop(paste("Error: Failed to write the file. Details:", e$message))
  })
}

In [ ]:
FindAllDEMs_2 <- function(data, ident, n = 3, sample_column = NULL, logFC_threshold = 1.2, DE_output_dir = NULL, run_name = "FindAllDEMs", annotation.column = NULL, assay = "Spatial", slot = "counts", return.individual = FALSE, verbose = TRUE, seed = 1234){

  if (!(is.null(DE_output_dir))){
    if (dir.exists(DE_output_dir)){
      warning("Please supply a directory path that doesn't already exist")
      stop("dir.exists(DE_output_dir) = TRUE")
    } else{
      dir.create(DE_output_dir)
    }
  }

  if (!is.factor(data@meta.data[[ident]])){
    stop("ident provided is not a factor! Please convert the ident column using `factor()` ...")
  }

  if (is.null(sample_column)){
      pooled_data <- run_pooling(data,ident, n = n, assay = assay, slot = slot, verbose = verbose, seed = seed)
  } else{
    if (!(sample_column %in% colnames(data@meta.data))){
      stop("sample_column provided is not found in the meta.data of the Seurat object!")
    }
    # make a new column in meta.data that combines ident and sample_column
    data@meta.data$combined <- paste(data@meta.data[[ident]], data@meta.data[[sample_column]], sep = "_")
    pooled_data <- run_pooling(data,"combined", n = n, assay = assay, slot = slot, verbose = verbose, seed = seed)
    n= n
  }


  #Step 2: Run EdgeR to calculate differentially expressed m/z peaks
  DEM_results <- run_DE(pooled_data, data, ident = ident, output_dir = DE_output_dir, run_name = run_name, n=n, logFC_threshold=logFC_threshold, annotation.column = annotation.column, assay = assay, verbose = verbose, return.individual = return.individual)

  # Returns an EDGEr object which contains the pseudo-bulk counts in $counts and DEMs in $DEMs
  return(DEM_results)
}

In [ ]:
#' Truncate and Sort Annotation Labels
#'
#' @description
#' Processes a vector of semicolon-separated string annotations. For each entry,
#' it sorts the labels based on their global frequency (case-insensitive) and then
#' by length, keeping only the top `n` labels.
#'
#' @details
#' The sorting logic is as follows:
#' 1.  **Primary Sort**: By the total number of occurrences of each label across the entire `annotation_column`. This is case-insensitive. More frequent labels come first.
#' 2.  **Tie-breaker**: If two labels have the same frequency, the one with fewer characters comes first.
#'
#' @param annotation_column A character vector where each element is a string of annotations separated by semicolons (e.g., `"Lipid A; GlcNAc; Peptidoglycan"`).
#' @param n An integer specifying the maximum number of sorted labels to keep for each entry. Defaults to `3`.
#'
#' @return A character vector of the same length as `annotation_column`, with each entry containing the sorted and truncated labels.
#'
#' @export
#' @examples
#' annotations <- c(
#'   "Short; LongLabel; Medium; Short",
#'   "LongLabel; Another; Medium",
#'   "VeryLongestLabel; Short"
#' )
#' # Global Frequencies (case-insensitive):
#' # short: 3
#' # longlabel: 2
#' # medium: 2
#' # another: 1
#' # verylongestlabel: 1
#'
#' # Tie-breaker (length):
#' # Medium (6) < LongLabel (9)
#'
#' # Final Order: Short, Medium, LongLabel, Another, VeryLongestLabel
#'
#' labels_to_show(annotations, n = 2)
#' # Expected output:
#' # [1] "Short; Medium"  (Short > Medium > LongLabel)
#' # [2] "Medium; LongLabel" (Medium and LongLabel are tied in freq, Medium is shorter)
#' # [3] "Short; VeryLongestLabel" (Short > VeryLongestLabel)

labels_to_show <- function(annotation_column, n = 3) {
  # --- Step 1: Calculate global frequency of all labels (case-insensitive) ---
  
  # Split all strings, unlist into a single vector, and clean up
  all_labels <- unlist(strsplit(annotation_column, ";\\s*"))
  all_labels <- all_labels[!is.na(all_labels) & all_labels != ""]
  
  # Create a frequency table from the lowercase versions of labels
  freq_table <- table(tolower(all_labels))
  
  # --- Step 2: Apply sorting and truncation to each entry ---
  
  new_column <- sapply(annotation_column, function(entry) {
    # Return empty string for NA or empty input
    if (is.na(entry) || nchar(trimws(entry)) == 0) {
      return("")
    }
    
    # Split the current entry's labels and clean them
    labels <- unlist(strsplit(entry, ";\\s*"))
    labels <- labels[!is.na(labels) & labels != ""]
    
    # Skip if no valid labels are found after cleaning
    if (length(labels) == 0) {
      return("")
    }
    
    # Get the frequency and length for each label
    # Use the pre-computed table for efficiency
    label_freqs <- as.numeric(freq_table[tolower(labels)])
    label_lengths <- nchar(labels)
    
    # Sort the original labels:
    # 1. By frequency in descending order (-label_freqs)
    # 2. By character length in ascending order (label_lengths)
    sorted_labels <- labels[order(-label_freqs, label_lengths)]
    
    # Select the top n labels and paste them back together
    top_n <- head(sorted_labels, n)
    paste(top_n, collapse = "; ")
  })
  
  return(new_column)
}
#' Heatmap of Differentially Expressed Metabolites
#'
#' Generates a heatmap of DEMs generated from edgeR analysis run using `FindAllDEMs()`.
#' This function uses `pheatmap` to plot data.
#'
#' @param edgeR_output A list containing outputs from edgeR analysis (from FindAllDEMs()). This includes pseudo-bulked counts and DEMs.
#' @param n An integer that defines the number of UP and DOWN regulated peaks to plot for each cluster (default = 5).
#' @param only.pos Boolean indicating if only positive markers should be returned (default = FALSE).
#' @param FDR.threshold Numeric value that defines the FDR threshold to use for filtering (default = 0.05).
#' @param logfc.threshold Numeric value that defines the logFC threshold for up/down regulation (default = 0.5).
#' @param order.by Character string defining which parameter to order markers by, options are 'FDR' or 'logFC' (default = "FDR").
#' @param scale A character string indicating if values should be scaled ("row", "column", "none").
#' @param color A vector of colors for the heatmap.
#' @param cluster_cols Boolean value to cluster columns (default = FALSE).
#' @param cluster_rows Boolean value to cluster rows (default = TRUE).
#' @param fontsize_row Numeric fontsize for row labels (default = 15).
#' @param fontsize_col Numeric fontsize for column labels (default = 15).
#' @param cutree_cols A numeric value for cutting the column tree into clusters (default = 9).
#' @param silent Boolean value indicating if the plot should not be drawn (default = TRUE).
#' @param plot_annotations_column Character string indicating the column name with metabolite annotations.
#' @param save_to_path Character string defining the full filepath to save the plot.
#' @param plot.save.width Integer for the width of the saved plot (default = 20).
#' @param plot.save.height Integer for the height of the saved plot (default = 20).
#' @param nlabels.to.show Numeric value for the number of annotations to show per m/z (default = NULL).
#' @param annotation_colors List for specifying annotation colors.
#'
#' @returns A pheatmap object.
#' @export
#'
#' @import dplyr
#' @import pheatmap
#' @import grDevices


DEMsHeatmap2 <- function(edgeR_output,
                        n = 5,
                        only.pos = FALSE,
                        FDR.threshold = 0.05,
                        logfc.threshold = 0.5,
                        order.by = "FDR",
                        scale = "row",
                        color = grDevices::colorRampPalette(c("navy", "white", "red"))(50),
                        cluster_cols = FALSE,
                        cluster_rows = TRUE,
                        fontsize_row = 15,
                        fontsize_col = 15,
                        cutree_cols = 9,
                        silent = TRUE,
                        plot_annotations_column = NULL,
                        save_to_path = NULL,
                        plot.save.width = 20,
                        plot.save.height = 20,
                        nlabels.to.show = NULL,
                        annotation_colors = NULL) {

  # --- 1. Filter and Select Top Markers ---
  
  degs <- edgeR_output$DEMs %>%
    filter(FDR < FDR.threshold)

  if (tolower(order.by) == "fdr") {
    top_pos <- degs %>%
      filter(logFC > logfc.threshold) %>%
      group_by(cluster) %>%
      slice_min(order_by = FDR, n = n, with_ties = FALSE)
      
    top_neg <- degs %>%
      filter(logFC < -logfc.threshold) %>%
      group_by(cluster) %>%
      slice_min(order_by = FDR, n = n, with_ties = FALSE)
      
  } else if (tolower(order.by) == "logfc") {
    top_pos <- degs %>%
      filter(logFC > logfc.threshold) %>%
      group_by(cluster) %>%
      slice_max(order_by = logFC, n = n, with_ties = FALSE)

    top_neg <- degs %>%
      filter(logFC < -logfc.threshold) %>%
      group_by(cluster) %>%
      slice_min(order_by = logFC, n = n, with_ties = FALSE)
  } else {
    stop("`order.by` must be either 'FDR' or 'logFC'")
  }

  if (only.pos) {
    top_markers_df <- top_pos
  } else {
    top_markers_df <- bind_rows(top_pos, top_neg)
  }
  
  # Get a final, unique list of genes to plot. This is the crucial fix.
  # We select genes based on their significance but ensure each gene appears only once.
  genes_to_plot <- top_markers_df %>%
    ungroup() %>%
    distinct(gene, .keep_all = TRUE) %>%
    arrange(cluster, desc(logFC)) # Optional: arrange for a cleaner final look in the data frame

  if (nrow(genes_to_plot) == 0) {
    stop("No significant markers found with the given thresholds. Cannot generate heatmap.")
  }


  # --- 2. Prepare Data Matrix for Heatmap ---
  
  # Get the log-CPM matrix for all genes
  cpm_matrix <- edgeR::cpm(edgeR_output, log = TRUE)
  
  # Subset the matrix to only include our selected genes.
  # The row order will match the order in `genes_to_plot$gene`. NO unique()!
  mtx <- cpm_matrix[genes_to_plot$gene, , drop = FALSE]

  
  # --- 3. Handle Annotations and Rownames ---
  
  heatmap_labels <- genes_to_plot$gene # Default to gene names
  
  if (!is.null(plot_annotations_column)) {
    if (!plot_annotations_column %in% colnames(edgeR_output$DEMs)) {
      warning(paste0("Annotation column '", plot_annotations_column, "' not found. Plotting m/z values instead."))
    } else {
      # Create a reliable mapping from gene ID to annotation
      label_map <- genes_to_plot %>%
        select(gene, !!sym(plot_annotations_column))
      
      # Handle multiple or truncated labels if requested
      if (!is.null(nlabels.to.show)) {
        # Assuming `labels_to_show` is a helper function you have
        label_map[[plot_annotations_column]] <- labels_to_show(label_map[[plot_annotations_column]], n = nlabels.to.show)
      }
      
      # Use the map to create the final heatmap labels, ensuring order is preserved
      heatmap_labels <- label_map[[plot_annotations_column]]
    }
  }
  
  # Assign the final, correctly-ordered labels to the matrix
  rownames(mtx) <- make.unique(heatmap_labels)


  # --- 4. Prepare Column Annotations ---
  
  col_annot <- data.frame(sample = edgeR_output$samples$ident)
  rownames(col_annot) <- colnames(mtx)


  # Prepare annotation colors if provided
  if (!is.null(annotation_colors)) {
    final_annotation_colors <- list(sample = unlist(annotation_colors))
  } else {
    final_annotation_colors <- NA
  }


  # --- 5. Generate and Save Heatmap ---
  
  p <- pheatmap::pheatmap(
    mtx,
    scale = scale,
    color = color,
    cluster_cols = cluster_cols,
    annotation_col = col_annot,
    cluster_rows = cluster_rows,
    fontsize_row = fontsize_row,
    fontsize_col = fontsize_col,
    cutree_cols = cutree_cols,
    silent = silent,
    annotation_colors = final_annotation_colors
  )

  if (!is.null(save_to_path)) {
    # Assuming `save_pheatmap_as_pdf` is a helper function you have
    save_pheatmap_as_pdf(pheatmap = p, filename = save_to_path, width = plot.save.width, height = plot.save.height)
  }

  return(p)
}

In [ ]:
library(plotly)
Plot3DFeature_fixed <- function(data,
                          features,
                          assays = c("SPT", "SPM"),
                          slots = "counts",
                          between.layer.height = 100,
                          names= NULL,
                          size = 3,
                          col.palette = c("Reds", "Blues"),
                          x.axis.label = "x",
                          y.axis.label = "y",
                          z.axis.label = "z",
                          show.x.ticks = FALSE,
                          show.y.ticks = FALSE,
                          show.z.ticks = FALSE,
                          show.image = NULL,
                          plot.height = 800,
                          plot.width = 1500,
                          image.sf = "lowres",
                          downscale.image = NULL,
                          color.min = NULL,
                          color.max = NULL,
                          reverse.color = c(FALSE, FALSE)  # New parameter to reverse color scales
){
  
  ## handeling of inncorect input legnths
  if (length(features) < 1 | length(features) > 2){
    stop("Number of features supplied does not match required length. Either 1 or 2 features must be supplied")
  }
  if (length(assays) > 2){
    stop("Number of assays supplied does not match required length. 2 assays or less must be supplied")
  }
  if (length(slots) > 2){
    stop("Number of slots supplied does not match required length. 2 slots or less must be supplied")
  }
  
  ## Handling of only 1 submitted value
  if (length(features) ==  1 ){
    features <- c(features, features)
  }
  if (length(assays) ==  1 ){
    assays <- c(assays, assays)
  }
  if (length(slots) ==  1 ){
    slots <- c(slots, slots)
  }
  if (length(col.palette) ==  1 ){
    col.palette <- c(col.palette[1], col.palette[1])
  }
  if (length(names) ==  1 ){
    names <- c(names, names)
  }
  if (length(reverse.color) == 1) {
    reverse.color <- c(reverse.color[1], reverse.color[1])
  }
  
  # Handle color scale limits
  if (!is.null(color.min) && length(color.min) == 1) {
    color.min <- c(color.min, color.min)
  }
  if (!is.null(color.max) && length(color.max) == 1) {
    color.max <- c(color.max, color.max)
  }
  
  feature_data <- list()
  
  i <- 1
  default_names <- c()
  for (feature in features) {
    
    if (feature %in% colnames(data@meta.data)){
      default_names <- c(default_names, feature)
      feature_data[[i]] <- data@meta.data[[feature]]
    } else {
      if (length(assays) != 0){
        
        feature_data[[i]] <- tryCatch({data[[assays[i]]][slots[i]][feature,]},
                                       error = function(err){
                                         stop("The feature provided does not exist in the ", assays[i],": ",slots[i], " object. Please provide a value feature")})
      } else {
        stop("No assay supplied. The feature ", feature," either does not exist in @meta.data slot, or no matching assay was provided for the gene/feature. Please check SpaMTP object")
      }
      default_names <- c(default_names, paste0(feature,"_", assays[i]))
    }
    i <- i + 1
  }
  if (!(is.null(names))){
    if (length(names) == 2) {
      default_names <- names
    } else {
      warning("length of names must be == 2. Default names will be used instead ... ")
    }
  }
  
  # Get coordinates
  if(exists("GetTissueCoordinates")) {
    coords_df <- GetTissueCoordinates(data)
  } else {
    coords_df <- data@images[[1]]@coordinates
  }
  
  # Ensure x and y columns exist and are numeric
  if(!"x" %in% colnames(coords_df) || !"y" %in% colnames(coords_df)) {
    if("imagerow" %in% colnames(coords_df) && "imagecol" %in% colnames(coords_df)) {
      coords_df$x <- as.numeric(coords_df$imagerow)
      coords_df$y <- as.numeric(coords_df$imagecol)
    } else if("row" %in% colnames(coords_df) && "col" %in% colnames(coords_df)) {
      coords_df$x <- as.numeric(coords_df$row)
      coords_df$y <- as.numeric(coords_df$col)
    }
  } else {
    coords_df$x <- as.numeric(coords_df$x)
    coords_df$y <- as.numeric(coords_df$y)
  }
  
  # Create scatter3d traces for each layer
  trace1 <- plotly::plot_ly(
    data = coords_df,
    x = coords_df$x,
    y = coords_df$y,
    z = rep(0 + between.layer.height, times = nrow(coords_df)),
    type = "scatter3d",
    mode = "markers",
    height = plot.height,
    width = plot.width,
    name = default_names[1],
    marker = list(color = feature_data[[1]],
                  coloraxis = 'coloraxis', 
                  size = size)
  )
  
  # Add second trace
  plot <- trace1 %>% 
    plotly::add_trace(
      data = coords_df,
      x = coords_df$x,
      y = coords_df$y,
      z = rep(0, times = nrow(coords_df)),
      type = "scatter3d",
      mode = "markers",
      name = default_names[2],
      marker = list(color = feature_data[[2]],
                    coloraxis = 'coloraxis2')
    )
  
  # Determine color scale limits
  cmin1 <- if(!is.null(color.min) && length(color.min) >= 1) color.min[1] else min(feature_data[[1]], na.rm = TRUE)
  cmax1 <- if(!is.null(color.max) && length(color.max) >= 1) color.max[1] else max(feature_data[[1]], na.rm = TRUE)
  cmin2 <- if(!is.null(color.min) && length(color.min) >= 2) color.min[2] else min(feature_data[[2]], na.rm = TRUE)
  cmax2 <- if(!is.null(color.max) && length(color.max) >= 2) color.max[2] else max(feature_data[[2]], na.rm = TRUE)
  
  # Function to create color scale (normal or reversed)
  create_colorscale <- function(palette_name, reverse = FALSE) {
    if (reverse) {
      # Reverse the color scale
      if (palette_name %in% c("Reds", "Blues", "Greens", "Greys", "Oranges", "Purples")) {
        # For single-hue sequential scales, add "_r" suffix doesn't work in plotly
        # So we need to manually reverse
        return(paste0(palette_name))  # Will handle with reversescale parameter
      } else {
        return(palette_name)
      }
    } else {
      return(palette_name)
    }
  }
  
  # Apply layout to the complete plot object with custom color scales
  plot <- plot %>%
    plotly::layout(
      autosize = F,
      height = plot.height,
      width = plot.width,
      scene = list(
        aspectmode = "data",
        xaxis = list(title = x.axis.label, showticklabels = show.x.ticks),
        yaxis = list(title = y.axis.label, showticklabels = show.y.ticks),
        zaxis = list(title = z.axis.label, showticklabels = show.z.ticks)
      ),
      coloraxis = list(
        cmin = cmin1,
        cmax = cmax1,
        colorbar = list(
          orientation = "v",
          xanchor = "right",
          x = 0,
          len = 0.5,
          title = list(side = "top", text = default_names[1])
        ),
        colorscale = col.palette[1],
        reversescale = reverse.color[1]  # Reverse the scale if needed
      ),
      coloraxis2 = list(
        cmin = cmin2,
        cmax = cmax2,
        colorbar = list(
          orientation = "v",
          xanchor = "left",
          len = 0.5,
          x = 0,
          title = list(side = "top", text = default_names[2])
        ),
        colorscale = col.palette[2],
        reversescale = reverse.color[2]  # Reverse the scale if needed
      )
    )
  
  if (!is.null(show.image)){
    
    # Get the image and scale factor from Seurat object
    img_obj <- data@images[[show.image]]
    color_matrix <- as.raster(img_obj@image)
    
    # Get the scale factor - this is crucial for alignment
    scale_factor <- img_obj@scale.factors[[image.sf]]
    
    # Create image pixel coordinates
    img_height <- nrow(color_matrix)
    img_width <- ncol(color_matrix)
    
    row_indices <- rep(seq_len(img_height), each = img_width)
    col_indices <- rep(seq_len(img_width), times = img_height)
    
    # Flatten the color matrix into a vector
    colors <- as.vector(color_matrix)
    
    # Create a data frame
    df <- data.frame(
      row = row_indices,
      col = col_indices,
      color = colors
    )
    
    if (!is.null(downscale.image)){
      grid_df <- df
      grid_size <- as.numeric(downscale.image)
      
      # Create a new column to identify grid cells
      grid_df$grid_row <- floor(grid_df$row / grid_size)
      grid_df$grid_col <- floor(grid_df$col / grid_size)
      
      # Aggregate points by grid cell
      df <- grid_df %>%
        dplyr::group_by(grid_row, grid_col) %>%
        dplyr::summarize(
          row = mean(row),
          col = mean(col),
          color = dplyr::first(color),
          .groups = "drop"
        )
    }
    
    # Use the scale factor from the Seurat object
    df$y <- df$col
    df$x <- df$row 
    
    plot <- plot %>% 
      plotly::add_trace(
        data = df,
        x = df$x,
        y = df$y,
        z = rep(0 - between.layer.height, times = nrow(df)),
        type = "scatter3d",
        mode = "markers",
        name = "H&E",
        marker = list(
          color = df$color,
          size = size * 0.5  # Make image points smaller
        )
      )
  }
  
  return(plot)
}


In [ ]:
plot_metabolite_pathway <- function(seurat_obj, metabolites, assay_name = "SPM", 
                                   pathway_name = "total_pathway", subset.type = NULL,
                                   return_plot = TRUE) {
  
  # Set the default assay
  DefaultAssay(seurat_obj) <- assay_name
  
  # Search for each metabolite and combine results
  metabolite_dfs <- list()
  
  for (i in seq_along(metabolites)) {
    metabolite_df <- SearchAnnotations(seurat_obj, metabolites[i], assay = assay_name)
    if (nrow(metabolite_df) > 0) {
      metabolite_dfs[[i]] <- metabolite_df
    } else {
      warning(paste("Metabolite", metabolites[i], "not found in assay"))
    }
  }
  
  # Check if any metabolites were found
  if (length(metabolite_dfs) == 0) {
    stop("No metabolites found in the specified assay")
  }
  
  # Combine all metabolite data frames
  pathway_metabolites <- do.call(rbind, metabolite_dfs)$mz_names
  
  # Subset the data - handle both single and multiple metabolites
  if (length(pathway_metabolites) == 1) {
    # For single metabolite, ensure it's still a matrix
    subset_data <- seurat_obj[[assay_name]]$data[pathway_metabolites, , drop = FALSE]
  } else {
    subset_data <- seurat_obj[[assay_name]]$data[pathway_metabolites, ]
  }
  
  # Calculate column sums - works for both single and multiple metabolites
  if (nrow(subset_data) == 1) {
    col_sums <- as.numeric(subset_data[1, ])
  } else {
    col_sums <- colSums(subset_data)
  }
  
  # Add to metadata
  seurat_obj@meta.data[[pathway_name]] <- col_sums
  
  # assign 0 to non-subset.type if subset.type is provided
  if (!is.null(subset.type)) {
    seurat_obj@meta.data <- seurat_obj@meta.data %>%
      mutate(!!pathway_name := ifelse(interface.classification == subset.type, 
                                     .data[[pathway_name]], 
                                     NA))
  }

  # Create the plot if requested
  if (return_plot) {
    plot <- SpatialPlot(seurat_obj, features = pathway_name, image.alpha = 0.2,keep.scale = "all",pt.size.factor=1.5)
    return(list(seurat_obj = seurat_obj, plot = plot, metabolite_names = pathway_metabolites))
  } else {
    return(list(seurat_obj = seurat_obj, metabolite_names = pathway_metabolites))
  }
}

In [ ]:
#' Plot two existing metadata features spatially
#'
#' This function visualizes two features from the Seurat object's metadata
#' on the same spatial plot. Feature 1 is shown as a larger outer circle,
#' and Feature 2 is shown as a smaller inner square.
#'
#' @param seurat_obj A Seurat object with spatial data.
#' @param feature1_name The column name in meta.data for the first feature (outer circle).
#' @param feature2_name The column name in meta.data for the second feature (inner square).
#' @param subset.type Optional: A value from 'interface.classification' to subset.
#'                    Other spots will be set to NA and colored grey.
#' @param feature1_colors Colors for the first feature's gradient.
#' @param feature2_colors Colors for the second feature's gradient.
#' @param point_size_feature1 Size of the outer circle (feature 1).
#' @param point_size_feature2 Size of the inner square (feature 2).
#' @param return_plot Boolean; whether to return the ggplot object.
#'
#' @return A list containing the (unmodified) seurat_obj, the ggplot object (if return_plot = TRUE),
#'         and the data frame used for plotting.
#'
plot_dual_metadata_spatial <- function(seurat_obj,
                                       feature1_name,
                                       feature2_name,
                                       subset.type = NULL,
                                       feature1_colors = c("lightblue", "yellow", "red"),
                                       feature2_colors = c("lightblue", "green", "black"),
                                       point_size_feature1 = 5,
                                       point_size_feature2 = 2,
                                       return_plot = TRUE) {

  # Check if features exist in metadata
  if (!feature1_name %in% colnames(seurat_obj@meta.data)) {
    stop(paste("Feature 1 '", feature1_name, "' not found in seurat_obj@meta.data", sep = ""))
  }
  if (!feature2_name %in% colnames(seurat_obj@meta.data)) {
    stop(paste("Feature 2 '", feature2_name, "' not found in seurat_obj@meta.data", sep = ""))
  }

  # Prepare data for plotting
  plot_data <- seurat_obj@meta.data
  plot_data$feature1_value <- plot_data[[feature1_name]]
  plot_data$feature2_value <- plot_data[[feature2_name]]

  # Handle subset.type if provided
  if (!is.null(subset.type)) {
    if (!"interface.classification" %in% colnames(plot_data)) {
        warning("`subset.type` provided but 'interface.classification' column not found. Ignoring subset.")
    } else {
        plot_data <- plot_data %>%
          mutate(
            feature1_value = ifelse(interface.classification == subset.type, feature1_value, NA),
            feature2_value = ifelse(interface.classification == subset.type, feature2_value, NA)
          )
    }
  }

  # Get spatial coordinates
  coordinates <- GetTissueCoordinates(seurat_obj)
  if(is.null(coordinates)) {
    stop("No tissue coordinates found. Use GetTissueCoordinates() or ensure spatial info is present.")
  }
  plot_data$x <- coordinates[rownames(plot_data), "imagerow"]
  plot_data$y <- coordinates[rownames(plot_data), "imagecol"]

  # Rotate coordinates for correct orientation (as in original function)
  plot_data$x <- max(plot_data$x, na.rm = TRUE) - plot_data$x + min(plot_data$x, na.rm = TRUE)
  temp <- plot_data$y
  plot_data$y <- plot_data$x
  plot_data$x <- temp

  # Normalize values to 0-1 range for consistent color scaling
  plot_data$feature1_norm <- scales::rescale(plot_data$feature1_value, to = c(0, 1))
  plot_data$feature2_norm <- scales::rescale(plot_data$feature2_value, to = c(0, 1))

  plot_list <- list(
    seurat_obj = seurat_obj,
    data = plot_data
  )

  if (return_plot) {
    # Create the plot
    p <- ggplot() +
      # Add feature 1 data as larger circles
      geom_point(
        data = plot_data,
        aes(x = x, y = y, color = feature1_norm),
        shape = 21,
        size = point_size_feature1
      ) +
      scale_color_gradientn(
        colours = feature1_colors,
        limits = c(0, 1),
        na.value = "grey90",
        name = paste(feature1_name, "\n(normalized)")
      ) +
      # Add feature 2 data as smaller squares
      geom_point(
        data = plot_data,
        aes(x = x, y = y, fill = feature2_norm),
        color = "white",
        shape = 22,
        size = point_size_feature2
      ) +
      scale_fill_gradientn(
        colours = feature2_colors,
        limits = c(0, 1),
        na.value = "grey90",
        name = paste(feature2_name, "\n(normalized)")
      ) +
      # Formatting
      coord_fixed() +
      theme_minimal() +
      theme(
        panel.background = element_blank(),
        plot.background = element_blank(),
        panel.border = element_blank(),
        axis.text = element_blank(),
        axis.ticks = element_blank(),
        axis.title = element_blank(),
        panel.grid = element_blank(),
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        plot.margin = margin(0.1, 0.1, 0.1, 0.1, "cm")
      ) +
      labs(title = paste("Spatial overlay:", feature1_name, "&", feature2_name))

    plot_list$plot <- p
  }

  return(plot_list)
}

In [ ]:
#' Plot a calculated metabolite score and an existing metadata feature spatially
#'
#' This function calculates a score for a list of metabolites and
#' visualizes it alongside an existing feature from the Seurat object's metadata
#' on the same spatial plot. The metabolite score is shown as a larger outer circle,
#' and the metadata feature is shown as a smaller inner square.
#'
#' @param seurat_obj A Seurat object with spatial data.
#' @param metabolites A character vector of metabolite names to search for.
#' @param metadata_feature_name The column name in meta.data for the second feature (inner square).
#' @param assay_metabolite The assay containing metabolite data.
#' @param pathway_name The name to store the calculated metabolite score in meta.data.
#' @param subset.type Optional: A value from 'interface.classification' to subset.
#'                    Other spots will be set to NA and colored grey.
#' @param metabolite_colors Colors for the metabolite score's gradient.
#' @param metadata_colors Colors for the metadata feature's gradient.
#' @param point_size_metabolite Size of the outer circle (metabolite score).
#' @param point_size_metadata Size of the inner square (metadata feature).
#' @param return_plot Boolean; whether to return the ggplot object.
#'
#' @return A list containing the modified seurat_obj, the ggplot object (if return_plot = TRUE),
#'         the names of metabolites found, and the data frame used for plotting.
#'
plot_metabolite_metadata_spatial <- function(seurat_obj,
                                             metabolites,
                                             metadata_feature_name,
                                             assay_metabolite = "SPM",
                                             pathway_name = "metabolite_pathway",
                                             subset.type = NULL,
                                             metabolite_colors = c("lightblue", "yellow", "red"),
                                             metadata_colors = c("lightblue", "green", "black"),
                                             point_size_metabolite = 5,
                                             point_size_metadata = 2,
                                             return_plot = TRUE) {

  # --- Process metabolites (from original function) ---
  DefaultAssay(seurat_obj) <- assay_metabolite

  metabolite_dfs <- list()
  for (i in seq_along(metabolites)) {
    metabolite_df <- SearchAnnotations(seurat_obj, metabolites[i], assay = assay_metabolite)
    if (nrow(metabolite_df) > 0) {
      metabolite_dfs[[i]] <- metabolite_df
    } else {
      warning(paste("Metabolite", metabolites[i], "not found in assay"))
    }
  }

  if (length(metabolite_dfs) == 0) {
    stop("No metabolites found in the specified assay")
  }

  pathway_metabolites <- do.call(rbind, metabolite_dfs)$mz_names
  
  if (length(pathway_metabolites) == 1) {
    subset_data <- seurat_obj[[assay_metabolite]]$data[pathway_metabolites, , drop = FALSE]
  } else {
    subset_data <- seurat_obj[[assay_metabolite]]$data[pathway_metabolites, ]
  }

  if (nrow(subset_data) == 1) {
    metabolite_scores <- as.numeric(subset_data[1, ])
  } else {
    metabolite_scores <- colSums(subset_data)
  }

  seurat_obj@meta.data[[pathway_name]] <- metabolite_scores

  # --- Check for metadata feature ---
  if (!metadata_feature_name %in% colnames(seurat_obj@meta.data)) {
    stop(paste("Metadata feature '", metadata_feature_name, "' not found in seurat_obj@meta.data", sep = ""))
  }

  # --- Handle subset.type for both features ---
  if (!is.null(subset.type)) {
     if (!"interface.classification" %in% colnames(seurat_obj@meta.data)) {
        warning("`subset.type` provided but 'interface.classification' column not found. Ignoring subset.")
    } else {
        seurat_obj@meta.data <- seurat_obj@meta.data %>%
          mutate(
            !!pathway_name := ifelse(interface.classification %in% subset.type, .data[[pathway_name]], NA),
            !!metadata_feature_name := ifelse(interface.classification %in% subset.type, .data[[metadata_feature_name]], NA)
          )
    }
  }

  # --- Prepare data for plotting ---
  plot_data <- seurat_obj@meta.data
  plot_data$metabolite_value <- plot_data[[pathway_name]]
  plot_data$metadata_value <- plot_data[[metadata_feature_name]]

  # Get spatial coordinates
  coordinates <- GetTissueCoordinates(seurat_obj)
   if(is.null(coordinates)) {
    stop("No tissue coordinates found. Use GetTissueCoordinates() or ensure spatial info is present.")
  }
  plot_data$x <- coordinates[rownames(plot_data), "imagerow"]
  plot_data$y <- coordinates[rownames(plot_data), "imagecol"]

  # Rotate coordinates for correct orientation
  plot_data$x <- max(plot_data$x, na.rm = TRUE) - plot_data$x + min(plot_data$x, na.rm = TRUE)
  temp <- plot_data$y
  plot_data$y <- plot_data$x
  plot_data$x <- temp

  # Normalize values to 0-1 range
  plot_data$metabolite_norm <- scales::rescale(plot_data$metabolite_value, to = c(0, 1))
  plot_data$metadata_norm <- scales::rescale(plot_data$metadata_value, to = c(0, 1))

  plot_list <- list(
    seurat_obj = seurat_obj,
    metabolite_names = pathway_metabolites,
    data = plot_data
  )

  if (return_plot) {
    # Create the plot
    p <- ggplot() +
      # Add metabolite data as larger circles
      geom_point(
        data = plot_data,
        aes(x = x, y = y, color = metabolite_norm),
        shape = 21,
        size = point_size_metabolite
      ) +
      scale_color_gradientn(
        colours = metabolite_colors,
        limits = c(0, 1),
        na.value = "grey90",
        name = paste(pathway_name, "\n(normalized)")
      ) +
      # Add metadata feature as smaller squares
      geom_point(
        data = plot_data,
        aes(x = x, y = y, fill = metadata_norm),
        color = "white",
        shape = 22,
        size = point_size_metadata
      ) +
      scale_fill_gradientn(
        colours = metadata_colors,
        limits = c(0, 1),
        na.value = "grey90",
        name = paste(metadata_feature_name, "\n(normalized)")
      ) +
      # Formatting
      coord_fixed() +
      theme_minimal() +
      theme(
        panel.background = element_blank(),
        plot.background = element_blank(),
        panel.border = element_blank(),
        axis.text = element_blank(),
        axis.ticks = element_blank(),
        axis.title = element_blank(),
        panel.grid = element_blank(),
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        plot.margin = margin(0.1, 0.1, 0.1, 0.1, "cm")
      ) +
      labs(title = paste("Spatial overlay:", pathway_name, "&", metadata_feature_name))

    plot_list$plot <- p
  }

  return(plot_list)
}

In [ ]:
plot_metabolite_gene_spatial <- function(seurat_obj, 
                                       metabolites, 
                                       genes,
                                       assay_metabolite = "SPM",
                                       assay_gene = "SPT",
                                       pathway_name = "metabolite_pathway",
                                       genes_name = "gene_expression",
                                       subset.type = NULL,
                                       metabolite_colors = c("lightblue","yellow", "red"),
                                       gene_colors = c("lightblue","green", "black"),
                                       point_size_metabolite = 5,
                                       point_size_gene = 2,
                                       return_plot = TRUE) {
  
  # Process metabolites first
  DefaultAssay(seurat_obj) <- assay_metabolite
  
  # Search for each metabolite and combine results
  metabolite_dfs <- list()
  
  for (i in seq_along(metabolites)) {
    metabolite_df <- SearchAnnotations(seurat_obj, metabolites[i], assay = assay_metabolite)
    if (nrow(metabolite_df) > 0) {
      metabolite_dfs[[i]] <- metabolite_df
    } else {
      warning(paste("Metabolite", metabolites[i], "not found in assay"))
    }
  }
  
  # Check if any metabolites were found
  if (length(metabolite_dfs) == 0) {
    stop("No metabolites found in the specified assay")
  }
  
  
  # Combine all metabolite data frames
  pathway_metabolites <- do.call(rbind, metabolite_dfs)$mz_names
  
  # Subset the metabolite data
  if (length(pathway_metabolites) == 1) {
    subset_data <- seurat_obj[[assay_metabolite]]$data[pathway_metabolites, , drop = FALSE]
  } else {
    subset_data <- seurat_obj[[assay_metabolite]]$data[pathway_metabolites, ]
  }
  
  # Calculate metabolite pathway scores
  if (nrow(subset_data) == 1) {
    metabolite_scores <- as.numeric(subset_data[1, ])
  } else {
    metabolite_scores <- colSums(subset_data)
  }
  
  # Add to metadata
  seurat_obj@meta.data[[pathway_name]] <- metabolite_scores
  
  # Handle subset.type if provided
  if (!is.null(subset.type)) {
    seurat_obj@meta.data <- seurat_obj@meta.data %>%
      mutate(!!pathway_name := ifelse(interface.classification == subset.type,
                                     .data[[pathway_name]],
                                     NA))
  }
  
  # Get gene expression data
  DefaultAssay(seurat_obj) <- assay_gene
    # Get the expression data for the genes
  # Check which genes are present in the data
  available_genes <- intersect(genes, rownames(seurat_obj[[assay_gene]]))
  
  if (length(available_genes) == 0) {
    stop("None of the specified genes were found in the assay")
  }
  
  if (length(available_genes) < length(genes)) {
    missing_genes <- setdiff(genes, available_genes)
    warning(paste("The following genes were not found:", 
                  paste(missing_genes, collapse = ", ")))
  }
  
  if (length(available_genes) >= 2) {
     # Subset the data for available genes
  subset_data <- seurat_obj[[assay_gene]]$data[available_genes, ]
  
  # Calculate column sums (total expression per spot/cell)
  col_sums <- colSums(subset_data)
  } else {
    # For single gene, ensure it's still a matrix
    subset_data <- seurat_obj[[assay_gene]]$data[available_genes, , drop = FALSE]
    col_sums <- as.numeric(subset_data[1, ])
  }
 
  
  # Add to metadata
  seurat_obj@meta.data[[genes_name]] <- col_sums
  
  # Assign 0 to non-subset.type if subset.type is provided
  if (!is.null(subset.type)) {
    seurat_obj@meta.data <- seurat_obj@meta.data %>%
      mutate(!!genes_name := ifelse(interface.classification == subset.type, 
                                       .data[[genes_name]], 
                                       NA))
  }
  
  # Prepare data for plotting
  plot_data <- seurat_obj@meta.data
  plot_data$metabolite_value <- plot_data[[pathway_name]]
  plot_data$gene_value <- plot_data[[genes_name]]
  
  # Get spatial coordinates
  coordinates <- GetTissueCoordinates(seurat_obj)
  plot_data$x <- coordinates[rownames(plot_data), "imagerow"]
  plot_data$y <- coordinates[rownames(plot_data), "imagecol"]
  # rotate coordinates for correct orientation
  plot_data$x <- max(plot_data$x) - plot_data$x + min(plot_data$x) # switching the cooridnates so it looks nice
    # Rotate coordinates 90 degrees (assuming origin is top-left, typical for images)
    temp <- plot_data$y
    plot_data$y <- plot_data$x
    plot_data$x <- temp

  
  # Normalize values to 0-1 range for consistent color scaling
  plot_data$metabolite_norm <- scales::rescale(plot_data$metabolite_value, to = c(0, 1))
  plot_data$gene_norm <- scales::rescale(plot_data$gene_value, to = c(0, 1))
  
  if (return_plot) {
    # Get the H&E image
    images <- Images(seurat_obj)
    if (length(images) == 0) {
      stop("No images found in Seurat object")
    }
    
    # Use the first image if multiple are present
    image_name <- images[1]
    image_data <- seurat_obj@images[[image_name]]
    
    
    # Create the plot with H&E background
    p <- ggplot() +

      # Add metabolite data as larger circles
      geom_point(
        data = plot_data,
        aes(x = x, y = y, color = metabolite_norm),
        shape = 21,
        size = point_size_metabolite,
       
      ) +
      scale_color_gradientn(
        colours = metabolite_colors,
        limits = c(0, 1),
         na.value = "grey90",
        name = paste(pathway_name, "\n(normalized)")
      ) +
      # Add gene expression as smaller squares
      geom_point(
        data = plot_data,
        aes(x = x, y = y, fill = gene_norm),
        color = "white",
        shape = 22,
        size = point_size_gene,
  
      ) +
      scale_fill_gradientn(
        colours = gene_colors,
        limits = c(0, 1),
         na.value = "grey90",
        name = paste(genes_name, "\n(normalized)")
      ) +
      # Formatting
      coord_fixed() +
      theme_minimal() +
      theme(
        panel.background = element_blank(),
        plot.background = element_blank(),
        panel.border = element_blank(),
        axis.text = element_blank(),
        axis.ticks = element_blank(),
        axis.title = element_blank(),
        panel.grid = element_blank(),
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        plot.margin = margin(0.1, 0.1, 0.1, 0.1, "cm")
      ) +
      labs(title = paste("Spatial overlay:", pathway_name, "&", genes_name))
    
    return(list(
      seurat_obj = seurat_obj,
      plot = p,
      metabolite_names = pathway_metabolites,
      data = plot_data
    ))
  } else {
    return(list(
      seurat_obj = seurat_obj,
      metabolite_names = pathway_metabolites,
      data = plot_data
    ))
  }
}


In [ ]:

# Function to create pseudo-replicates within each sample
create_pseudo_replicates <- function(data, n_replicates = 4, seed = 123) {
  set.seed(seed)  # for reproducibility
  pseudo_data <- data.frame()
  
  for (s in unique(data$sample)) {
    # Shuffle the data for this sample
    sample_data <- data %>% 
      filter(sample == s) %>%
      sample_frac(1)  # This shuffles all rows
    
    n_cells <- nrow(sample_data)
    
    # Calculate size of each split
    base_size <- n_cells %/% n_replicates
    remainder <- n_cells %% n_replicates
    
    # Split into pseudo-replicates
    start_idx <- 1
    for (i in 1:n_replicates) {
      # Add 1 extra to first 'remainder' splits to handle uneven division
      current_size <- base_size + (i <= remainder)
      end_idx <- start_idx + current_size - 1
      
      # Calculate mean for this pseudo-replicate
      pseudo_data <- rbind(pseudo_data, 
                          data.frame(
                            sample = s,
                            group = ifelse(s %in% c("T1", "T2"), "T", "C"),
                            replicate = paste0(s, "_rep", i),
                            metabolites = mean(sample_data$metabolites[start_idx:end_idx], na.rm = TRUE),
                            n_cells = current_size
                          ))
      
      start_idx <- end_idx + 1
    }
  }
  
  return(pseudo_data)
}


# Figure 4 joint pathway analysis

## redo metabolites annotation to get save intermidiate data base infomation

The input here is the intergrated SPaMTP data object with MALDI and Visium aligned.

In [ ]:
test_data <- readRDS(paste0(OUT_DIR,"data_objects/merged_MALDIVIS_with_clusters.RDS"))

In [ ]:
re_annotated <- AnnotateSM(test_data, HMDB_db,adducts = c("M+H"),
           assay = "SPM",    save.intermediate = TRUE)
re_annotated[["SPM"]]@meta.data <- RefineLipids(re_annotated[["SPM"]]@meta.data, annotation.column = "all_IsomerNames", lipid_info = "simple")



## define human and mouse regions

In [ ]:
re_annotated@meta.data$interface.classification <- case_when(
  re_annotated@meta.data$percent.human < 10 ~ "mouse",
  re_annotated@meta.data$percent.human >= 10 & re_annotated@meta.data$percent.human < 50 ~ "interface.mouse.like",
  re_annotated@meta.data$percent.human >= 50 & re_annotated@meta.data$percent.human < 90 ~ "interface.human.like",
  re_annotated@meta.data$percent.human >= 90 ~ "human"
)
re_annotated@meta.data$interface.classification <- case_when(
  re_annotated@meta.data$interface.classification == "interface.mouse.like" & 
    re_annotated@meta.data$SPM_Clusters %in% c("1","3","4","6") ~ "mouse",
  TRUE ~ re_annotated@meta.data$interface.classification
)

In [ ]:
saveRDS(re_annotated, file = "re_annotated.rds")

## DE analysis

focus on human genes only

In [ ]:
re_annotated <- keep_human_genes_only(re_annotated)

### for human region: treatment vs control

In [ ]:
processing_seurat_obj <- subset(re_annotated, subset = interface.classification %in% c("human","interface.human.like"))

# add random number between 1 to 30 as a new column in meta.data
set.seed(123) # For reproducibility
processing_seurat_obj@meta.data$pseudo_rep <- sample(1:30, size = nrow(processing_seurat_obj@meta.data), replace = TRUE)
colnames(processing_seurat_obj@meta.data)
table(processing_seurat_obj@meta.data$pseudo_rep)
pseudo_bulk <- AggregateExpression(processing_seurat_obj, group.by = c("treatment","sample","pseudo_rep"), return.seurat = TRUE)
pseudo_bulk@tools$db_3 <- processing_seurat_obj@tools$db_3


Idents(pseudo_bulk) <- "treatment"
deg =  FindAllMarkers(pseudo_bulk,  assay = "SPT",only.pos =FALSE)
dem =  FindAllMarkers(pseudo_bulk, assay = "SPM", only.pos =FALSE)

#DE.list <- list("genes" = deg %>%  dplyr::filter(p_val_adj < 0.05 ) ,"metabolites" = dem%>%  dplyr::filter(p_val_adj < 0.05))
DE.list <- list("genes" = deg  ,"metabolites" = dem)

## Run multi-modal pathway analysis
regpathway <- FindRegionalPathways(
  pseudo_bulk,
  analyte_types = c("genes", "metabolites"),
  SM_slot = "data", ST_slot = "data",
  ident = "treatment",
  DE.list = DE.list,
    pval_cutoff_mets = 999,
pval_cutoff_genes = 999,
    all_genes=TRUE
)



In [ ]:
# keep only the first row if pathwayNames are duplicated
pathways_1 <-regpathway %>% filter(Cluster_id=="Treated") %>% arrange((-NES)) %>% distinct(pathwayName, .keep_all = TRUE) %>% head(6) 
pathways_2 <-regpathway %>% filter(Cluster_id=="Control") %>% arrange((-NES)) %>% distinct(pathwayName, .keep_all = TRUE) %>% head(6)
pathway_targets <- rbind(pathways_1, pathways_2)$pathwayName
# write to xlsx
library(openxlsx)
write.xlsx(regpathway, file = "/scratch/user/uqyche42/projects/SM_mouse_brain/regpathway_human.xlsx", sheetName = "treatment", overwrite = TRUE)
PlotRegionalPathways(regpathway = regpathway,selected_pathways=pathway_targets,num_display=12)

In [ ]:
regpathway %>% 
  filter(Cluster_id == "Treated") %>% 
  # Fix 1: Use grepl to find rows containing ";"
  filter(grepl(";", rna_regulation)) %>% 
    filter(grepl(";", adduct_info)) %>% 
    filter(padj<0.05) %>%
  # Fix 2: Use desc() for sorting (cleaner than -NES)
  arrange(NES) %>% 
  # Fix 3: TRUE keeps the rest of the columns (the row with the highest NES)
  # Fix 4: FALSE must be uppercase
  distinct(pathwayName, .keep_all = TRUE)

THis should remove most of the redundant pathway entries from different sources. Here are the manually curated pathways to make sure that:
    1. No redundant pathways
    2. All pathways have padj<0.05
    3. All pathways have at least two leading edges in RNAs and in metabolites (i.e. DEG and DEM)

In [ ]:
pathway_targets<-c("Transcriptional Regulation by TP53",
"Metabolic reprogramming in colon cancer",
"Regulation of Glucokinase by Glucokinase Regulatory Protein",
"Glycolysis / Gluconeogenesis",
"Trans-sulfuration, one-carbon metabolism and related pathways",
"Glucose metabolism",
"Metabolic Epileptic Disorders",
"Platelet homeostasis",
"Metabolism of steroids",
"Amino acid metabolism",
"Metabolism of vitamins and cofactors",
"Peptide hormone metabolism",
"Sphingolipid pathway")

In [ ]:
#set figure size
options(repr.plot.width = 15, repr.plot.height = 10)
plot <-PlotRegionalPathways(regpathway = regpathway,selected_pathways=pathway_targets,num_display=20)
plot
# save plot to pdf
ggsave(
  filename = "pathways_joined.pdf",  # The name of the file to save
  plot = plot,                  # The plot object you created
  device = "pdf",                    # Explicitly state the file type
  width = 15,                         # Width in inches
  height = 10,                        # Height in inches
  units = "in",                      # Unit of measurement
  dpi = 300                          # Resolution (300 is standard for print)
)

 This gives the network in html format

In [ ]:

PathwayNetworkPlots(processing_seurat_obj,ident = "treatment", regpathway =  regpathway, DE.list = DE.list , selected_pathways = pathway_targets, path = "Pathway_Network")


3D plotting Examples 

In [ ]:
obj.list <- SplitObject(re_annotated, split.by = "sample")

hg38-KIF5C and mz-175.118662442112 are key nodes from "Peptide hormone metabolism"

In [ ]:
Plot3DFeature_fixed(obj.list[["T2"]], 
              features = c("hg38-KIF5C", "mz-175.118662442112"), 
              assays = c("SPT", "SPM"), 
              between.layer.height = 250, 
              show.image = "slice1.3", 
              plot.height = 600, 
              plot.width = 1200,color.min = c(0, 0),      # Minimum values for both features
                    color.max = c(4, 400),
                   reverse.color = c(FALSE, TRUE)) 

In [ ]:
Plot3DFeature_fixed(obj.list[["C2"]], 
              features = c("hg38-KIF5C", "mz-175.118662442112"), 
              assays = c("SPT", "SPM"), 
              between.layer.height = 250, 
              show.image = "slice1.4", 
              plot.height = 600, 
              plot.width = 1200,color.min = c(0, 0),      # Minimum values for both features
                    color.max = c(4, 400), 
                    reverse.color = c(FALSE, TRUE)) 

# Figure 5

In [ ]:
# FIgure 5b

options(repr.plot.width = 15, repr.plot.height = 15) 
col_palette = list("1" = "#9FBEAC", 
                   "4" = "#C2B03B",
                   "3" = "#F99D1D",
                   "6" = "#008E87", 
                   "5" = "#0074B0",  
                   "2" = "#DE4D6C", 
                   "7" = "#F99D1D",
                   "8" = "#DE4D6C",
                   "0" = "#BF212E")
ImageDimPlot(re_annotated, group.by = "seurat_clusters", cols = col_palette,size = 2, fov = c("fov","fov.2","fov.3","fov.4")) + coord_flip() + scale_x_reverse()

In [ ]:
# Figure 5c
options(repr.plot.width = 20, repr.plot.height = 20)
spatial_map <- SpatialPlot(
  re_annotated, 
  group.by = "interface.classification", 
  images = c("slice1", "slice1.3")
)


ggsave(
  filename = "figures_publication/Spatial_interface_1.pdf",
  plot = spatial_map,
  width = 12,       # Spatial plots often need more width, especially with 2 slices
  height = 6,
  device = "pdf"
)


spatial_map <- SpatialPlot(
  re_annotated, 
  group.by = "interface.classification", 
  images = c("slice1.2", "slice1.4")
)


ggsave(
  filename = "figures_publication/Spatial_interface_2.pdf",
  plot = spatial_map,
  width = 12,       # Spatial plots often need more width, especially with 2 slices
  height = 6,
  device = "pdf"
)

# Figure 5d
options(repr.plot.width = 20, repr.plot.height = 20)

# use test_data, the original unfiltered data as we need the mouse gene here
SpatialFeaturePlot(test_data, features = c("mm10-Gfap"))



In [ ]:
# figure 5e and f

interface <- subset(re_annotated, subset = interface.classification %in% c("interface.mouse.like","interface.human.like"))
interface <- subset(interface, subset = treatment %in% c("Control")) 
interface_DEMs <- FindAllDEMs(interface, "interface.classification",assay="SPM", n = 12, logFC_threshold = 1.2, DE_output_dir = "DEM_cluster_interface" ,
                           run_name = "FindAllDEPs", 
                           annotation.column = "all_IsomerNames")
voca_plot_data <- interface_DEMs$DEMs[interface_DEMs$DEMs$cluster == "interface.mouse.like", ]
EnhancedVolcano::EnhancedVolcano(voca_plot_data, lab = voca_plot_data$annotations, x = "logFC", y = "FDR",pCutoff = 0.05,FCcutoff = 1.2)

# keep all results as required for MetaboAnalyst 
export_regulation_xlsx(voca_plot_data, 
                        fdr_threshold = 100000, 
                        logfc_threshold = 0.0000001, 
                        filename = "All_SM_contrtol_interface_mouse_like_regulation_results.xlsx")

# run MetaboAnalyst on website using this file


In [ ]:
# figure 5h and i
options(repr.plot.width = 20, repr.plot.height = 20)
target <- c("N-Acetyl-L-aspartic acid")
name<- "N-Acetyl-L-aspartic acid"

plot_metabolite_metadata_spatial(obj.list[["C1"]],target,"human.like",
                                 subset.type=c("interface.human.like","interface.mouse.like"),pathway_name=name,
                                              metabolite_colors = c("lightblue", "red"),
                                             metadata_colors = c("lightblue", "black"),
                                point_size_metabolite = 6)$plot

plot_metabolite_metadata_spatial(obj.list[["C2"]],target,"human.like",
                                 subset.type=c("interface.human.like","interface.mouse.like"),pathway_name=name,
                                              metabolite_colors = c("lightblue", "red"),
                                             metadata_colors = c("lightblue", "black"),
                                point_size_metabolite = 6)$plot

options(repr.plot.width = 20, repr.plot.height = 20)

name<- "Ganglioside GM2"
target <- c("Ganglioside GM2")
#plot_metabolite_metadata_spatial(obj.list[["T1"]],target,"human.like",
 #                                subset.type=c("interface.human.like","interface.mouse.like"),pathway_name=name,
                                            #  metabolite_colors = c("lightblue", "red"),
                                          #   metadata_colors = c("lightblue", "black"),
  #                              point_size_metabolite = 6)$plot

#plot_metabolite_metadata_spatial(obj.list[["T2"]],target,"human.like",
#                                 subset.type=c("interface.human.like","interface.mouse.like"),pathway_name=name,
                                           #   metabolite_colors = c("lightblue", "red"),
                                          #   metadata_colors = c("lightblue", "black"),
#                                point_size_metabolite = 6)$plot
plot_metabolite_metadata_spatial(obj.list[["C1"]],target,"human.like",
                                 subset.type=c("interface.human.like","interface.mouse.like"),pathway_name=name,
                                              metabolite_colors = c("lightblue", "red"),
                                             metadata_colors = c("lightblue", "black"),
                                point_size_metabolite = 6)$plot

plot_metabolite_metadata_spatial(obj.list[["C2"]],target,"human.like",
                                 subset.type=c("interface.human.like","interface.mouse.like"),pathway_name=name,
                                              metabolite_colors = c("lightblue", "red"),
                                             metadata_colors = c("lightblue", "black"),
                                point_size_metabolite = 6)$plot                               

# Figure 6

## comparing  treatment and control for human-like interface

In [ ]:
interface <- subset(test_data, subset = interface.classification %in% c("interface.human.like"))
interface <- subset(interface, subset = treatment %in% c("Treated","Control"))

In [ ]:
# figure 6a
interface_DEMs <- FindAllDEMs(interface, "treatment",assay="SPM", n = 12, logFC_threshold = 1.2, DE_output_dir = NULL ,
                           run_name = "FindAllDEPs", 
                           annotation.column = "all_IsomerNames")
                           DEMsHeatmap(interface_DEMs,only.pos = FALSE,order.by = "logFC",
            plot_annotations_column = "annotations", nlabels.to.show = 1, 
            n = 40, save_to_path = NULL
            ,plot.save.height = 10, plot.save.width = 15)
voca_plot_data <- interface_DEMs$DEMs[interface_DEMs$DEMs$cluster == "Treated", ]
export_regulation_xlsx(voca_plot_data, 
                        fdr_threshold = 0.05, 
                        logfc_threshold = 1.2, 
                        filename = "treated_vs_control_for_interface_mouse.xlsx")
EnhancedVolcano::EnhancedVolcano(voca_plot_data, lab = voca_plot_data$annotations, x = "logFC", y = "FDR",pCutoff = 0.05,FCcutoff = 1.2)
voca_plot_data <- interface_DEMs$DEMs[interface_DEMs$DEMs$cluster == "Treated", ]
export_regulation_xlsx(voca_plot_data, 
                        fdr_threshold = 100000000, 
                        logfc_threshold = 0.0000001, 
                        filename = "All_SM_treated_vs_control_for_interface_mouse.xlsx")
# run MetaboAnalyst on website using this file                        

In [ ]:
# figure 6b
library(patchwork)
# key metabolites from MetaboAnalyst analysis
metabolites <- c("1-Methylhistamine", "Methylimidazoleacetic acid", "Histidine", "5'-Phosphoribosyl-N-formylglycinamide", "7,8-Dihydropteroic acid", "Pantothenic acid", "Lysine")


result <- plot_metabolite_pathway(test_data, c(metabolites), subset.type = "interface.human.like", pathway_name= "metabolites")

# Access the updated Seurat object
test_data <- result$seurat_obj

# Display the plot
p <-result$plot
# Extract the data range from all plots to find global min/max
p_modified <- lapply(p, function(x) {
  x + scale_fill_viridis_c(option = "inferno", na.value = "grey90",
                          limits = c(0, 250)
                         )
})

# Combine them back
metabolites_plot<-wrap_plots(p_modified, ncol = 2)
ggsave(
  filename = 'figures_publication/metabolites_inferface_spatial2.pdf',
  plot = metabolites_plot,
  width = 20,    # Adjust width (inches)
  height = 20,    # Adjust height (inches)
  device = "pdf"
)


# Create pseudo-replicates
#pseudo_replicates <- create_pseudo_replicates(subset, n_replicates = 6, seed = sample(1:100, 1))
pseudo_replicates <- create_pseudo_replicates(plotting_table, n_replicates = 6, seed = 8)

# Perform statistical test (t-test)
t_test_result <- t.test(metabolites ~ group, data = pseudo_replicates)
print(t_test_result)

# Wilcoxon test (non-parametric alternative)
wilcox_result <- wilcox.test(metabolites ~ group, data = pseudo_replicates)
print(wilcox_result)

# Visualization with statistical test
p1 <- ggboxplot(pseudo_replicates, 
                x = "group", 
                y = "metabolites",
                color = "group",
                palette = c("T" = "#00AFBB", "C" = "#FC4E07"),
                add = "jitter",
                shape = "group") +
  stat_compare_means(method = "wilcox.test", label.x = 1.5, label.y = max(pseudo_replicates$metabolites) * 1.1) +
  labs(title = "metabolites: T (T1+T2) vs C (C1+C2)",
       subtitle = "6 pseudo-replicates per group",
       y = "metabolites") +
  theme_minimal()

print(p1)

ggsave(
  filename = 'figures_publication/metabolites_inferface_boxplot.pdf',
  plot = p1,
  width = 5,    # Adjust width (inches)
  height = 5,    # Adjust height (inches)
  device = "pdf"
)

# Figure 7

## identify drug positive spots and proliferating spots

We are doing background subtraction here. The assumption is that the this drug concentration threshold is high so that none of the MALDI pixels in control is positive. Then any Visium spots contain positive pixel would be considered drug positive.

In [ ]:
# process from the MALDI data to add drug detection status
maldi_data <-readRDS(paste0(OUT_DIR,"data_objects/all_data_annotated.RDS"))
max_drug_control <- max(
  GetAssayData(maldi_data[["Spatial"]], slot = "data")["mz-448.246729024761", 
    which(maldi_data@meta.data$treatment == "Control")]
  )
maldi_data@meta.data$drug_detected <- ifelse(
  GetAssayData(maldi_data[["Spatial"]], slot = "data")["mz-448.246729024761", ] > max_drug_control,
  "Positive",
  "Negative"
)
# save metadata with rownames
# 1. Access the meta.data slot
metadata_df <- maldi_data@meta.data

# 2. Convert Rownames to a Column
# We use tibble::rownames_to_column() or base R to turn the index into a column named 'Cell_ID' (or similar).
library(tibble)
metadata_df <- rownames_to_column(metadata_df, var = "Cell_ID")

# OR using base R:
# metadata_df$Cell_ID <- rownames(metadata_df)
# rownames(metadata_df) <- NULL # Optional: removes the row names from the data frame itself

# 3. Save the Data Frame to CSV
write.csv(metadata_df, 
          file = "maldi_metadata_with_ids.csv",
          row.names = FALSE) # Set to FALSE since the IDs are now a column

In [ ]:
maldi_metadata<-read.csv("maldi_metadata_with_ids.csv")
positive_pixel <- maldi_metadata[maldi_metadata$drug_detected=="Positive",]$Cell_ID

In [ ]:
library(dplyr)
library(stringr)



# Use the pipe to modify the @meta.data slot
re_annotated@meta.data <- re_annotated@meta.data %>%
  mutate(
    # Check if any positive pixel pattern is found in the MALDI_barcodes string.
    # str_detect returns TRUE if a match is found, FALSE otherwise.
    drug_detected = if_else(
      str_detect(MALDI_barcodes, pattern = pixel_pattern),
      "Positive",
      "Negative"
    )
  )
  # set figure size
options(repr.plot.width = 32, repr.plot.height = 8)
SpatialDimPlot(
  re_annotated,
  group.by = "drug_detected"
)

In [ ]:
re_annotated@meta.data$drug_prolif_combined <- paste0(
  re_annotated@meta.data$drug_detected, "_",
  ifelse(re_annotated@meta.data$HALLMARK_E2F_TARGETS.enrich.scores >0.5, 
         "prolif", "nonprolif")
)
re_annotated@meta.data$drug_prolif_combined <- paste0(
  re_annotated@meta.data$drug_detected, "_",
  ifelse(re_annotated@meta.data$HALLMARK_E2F_TARGETS.enrich.scores >0.5, 
         "prolif", "nonprolif")
)

# Define custom colors with transparency
unique_drug <- unique(re_annotated@meta.data$drug_detected)


all_colors <- all_colors <- c(
  "Negative_prolif" = "#FF0000",     # Bright red for Negative interface
  "Positive_prolif" = "#0066FF",     # Bright blue for Positive interface
  "Negative_nonprolif" = "#FFB3B3",         # Light/pale red for Negative other
  "Positive_nonprolif" = "#B3D1FF"          # Light/pale blue for Positive other
)

# Create plot with custom colors
p <- SpatialDimPlot(re_annotated, 
                   group.by = "drug_prolif_combined",
                   cols = all_colors) +
  ggtitle("Drug Detection with proliferation Highlighting")

print(p)
interface_subset <- subset(
  re_annotated,
  subset = interface.classification == "interface.human.like"
)
p <- SpatialDimPlot(interface_subset, 
                   group.by = "drug_prolif_combined",
                   cols = all_colors) +
  ggtitle("Drug Detection with proliferation Highlighting")

print(p)
re_annotated@meta.data$drug_interface_combined <- paste0(
  re_annotated@meta.data$drug_detected, "_",
  ifelse(re_annotated@meta.data$interface.classification == "interface.human.like", 
         "interface", "other")
)

# Define custom colors with transparency
unique_drug <- unique(re_annotated@meta.data$drug_detected)

# Figure 7a
all_colors <- all_colors <- c(
  "Negative_interface" = "#FF0000",     # Bright red for Negative interface
  "Positive_interface" = "#0066FF",     # Bright blue for Positive interface
  "Negative_other" = "#FFB3B3",         # Light/pale red for Negative other
  "Positive_other" = "#B3D1FF"          # Light/pale blue for Positive other
)

# Create plot with custom colors
p <- SpatialDimPlot(re_annotated, 
                   group.by = "drug_interface_combined",
                   cols = all_colors) +
  ggtitle("Drug Detection with Interface Highlighting")

print(p)
table(re_annotated@meta.data$drug_prolif_combined)

## comparing resist vs not resist

In [ ]:
test_data <- subset(re_annotated, subset = drug_prolif_combined %in% c("Positive_prolif", "Positive_nonprolif"))
test_data<- subset(test_data,subset = sample %in% c("T1","T2"))
test_data<- subset(test_data,subset = interface.classification %in% c("interface.human.like","human"))

test_DEG<-FindMarkers(
  test_data,
  ident.1 = "Positive_prolif",
  ident.2 = "Positive_nonprolif",

  group.by = "drug_prolif_combined",

  assay = "SPT",
    only.pos=FALSE

)

library(stringr)
test_DEG <- test_DEG %>%
  tibble::rownames_to_column(var = "gene") %>%
  

  # The pattern "^hg38" means "starts with hg38"
  filter(str_detect(gene, "^hg38")) %>%
  
  tibble::column_to_rownames(var = "gene")

  # Load necessary libraries
library(ggplot2)
library(dplyr)
library(tibble)



# Define thresholds
FC_THRESHOLD <- 1.0
P_ADJ_THRESHOLD <- 0.05

top_genes_to_label <- rownames(head(test_DEG, 10))

# Figure 7d
volcano_plot <-EnhancedVolcano(
       xlim = c(-5, 10),
    ylim = c(0,10),
  toptable = test_DEG,
  lab = rownames(test_DEG),
  x = 'avg_log2FC',
  y = 'p_val_adj',
  # Thresholds for coloring and labeling
  pCutoff = P_ADJ_THRESHOLD,
  FCcutoff = FC_THRESHOLD,
  # Labeling options
  selectLab = top_genes_to_label, 
  boxedLabels = TRUE,
  # Customization
  pointSize = 6.0,
  labSize = 9.0,
  subtitle = paste0('FC Cutoff: ', FC_THRESHOLD, ' | P-adj Cutoff: ', P_ADJ_THRESHOLD),
  # Save to file
  drawConnectors = TRUE,

)
ggsave(
  filename = 'figures_publication/Volcano_resist.pdf',
  plot = volcano_plot,
  width = 10,    # Adjust width (inches)
  height = 8,    # Adjust height (inches)
  device = "pdf"
)


# save to csv
write.csv(test_DEG,"DEG_drug_resis.csv",quote=FALSE)
sig_test_DEG <- test_DEG%>% filter(p_val_adj<=0.05)

hg38_genes <- sig_test_DEG %>%
  filter(grepl("^hg38-", X)) %>%
  pull(X)

# Extract gene expression data from the SPT assay
gene_expression <- GetAssayData(test_data, 
                                assay = "SPT", 
                                slot = "data")[hg38_genes, , drop = FALSE]

# Convert gene expression matrix to dataframe and transpose
gene_expr_df <- as.data.frame(t(as.matrix(gene_expression)))

# Extract metadata columns
metadata_df <- test_data@meta.data[, c("drug_prolif_combined","HALLMARK_E2F_TARGETS.enrich.scores", 
                                        "drug_concentration")]

# Ensure row names match
gene_expr_df <- gene_expr_df[rownames(metadata_df), , drop = FALSE]

# Combine metadata and gene expression into one dataframe
final_df <- cbind(metadata_df, gene_expr_df)

# View the structure of the final dataframe
str(final_df)
head(final_df)

write.csv(final_df,"drug_resis_data.csv",quote=FALSE)
# drug_resis_data.csv used for GSEA analysis via MetaboAnalyst to make figure 7c,e,f